# BayesRTMMRL vs BayesAdapter: MSP uncertainty, R/C feature analysis, and OOD evidence chain

本 notebook 是对原 `BayesRTMMRL_vs_BayesAdapter_OOD_rerun_summary` 的结构性改版，核心改动如下：

1. **主不确定性分数固定为 MSP uncertainty**：`1 - max softmax probability`。
2. **不再画 confidence histogram**，所有分布图使用 KDE/ECDF 曲线。
3. **每个 seed 单独作为一个 subplot**，不做 seed 聚合主图。
4. **BayesRTMMRL 的 R 分支和 C 分支都进入特征分析**：embedding、similarity heatmap、similarity distribution 都分别绘制。
5. **BayesRTMMRL 与 BayesAdapter 必须同图对比**，避免分开画导致不可比。
6. 输出包括 OOD detection、calibration、ID error detection、risk-coverage、feature similarity statistics。

本 notebook 只做 evaluation / analysis，不训练模型。

## 1. 环境初始化

这一步只加载项目模块和常用分析库。不会训练模型。

In [1]:

# =========================
# 0. 配置区
# =========================
from pathlib import Path

NOTEBOOK_VERSION = "msp_uncertainty_rc_analysis_v4_checked_2026_05_24"

REPO_ROOT = Path("/root/autodl-tmp/MMRL").expanduser().resolve()
DATASET_ROOT = REPO_ROOT / "DATASETS"
OUTPUT_ROOT = REPO_ROOT / "output_refactor"

ID_DATASETS = ["cifar_10"]
OOD_DATASETS = ["dtd", "tinyimagenet", "oxford_flowers", "sun397"]

PROTOCOL = "FS"
SHOTS = [8 ]
SEEDS = [1]
BACKBONE = "ViT-B/16"

# BayesRTMMRL -> online, BayesAdapter -> cache，与原脚本保持一致。
EXEC_MODE_BY_METHOD = {
    "BayesRTMMRL": "online",
    "BayesAdapter": "cache",
}

LOAD_EPOCH = None
OOD_BATCH_SIZE = 250
OOD_NUM_WORKERS = 4

# 缓存控制
RECOMPUTE_CACHE = True       # True: 即使已有缓存也重新 forward
SAVE_FEATURES = True         # 必须 True，否则无法画 R/C 分支和 similarity matrix
KEEP_ON_CPU = True

# 主分数：固定为 MSP uncertainty
MAIN_UNCERTAINTY_SCORE = "msp_uncertainty"
HISTOGRAM_ENABLED = False
DISTRIBUTION_PLOT_TYPE = "KDE"   # KDE 或 ECDF；主图默认 KDE，必要时可补 ECDF

# BayesRTMMRL 特征分支。会优先从 outputs.features 中找这些 key 的候选名。
FEATURE_BRANCHES = {
    "BayesRTMMRL": ["R", "C"],
    "BayesAdapter": ["adapter"],
}

# 不同项目实现中，features 字典的 key 可能不同。按顺序匹配。
FEATURE_KEY_CANDIDATES = {
    "R": ["R", "r", "rep", "feat_R", "features_R", "image_features_R", "image_features_rep", "r_features", "robust", "robust_features", "z_R", "r_branch"],
    "C": ["C", "c", "main", "img", "image", "feat_C", "features_C", "image_features_C", "image_features_main", "c_features", "class", "class_features", "z_C", "c_branch"],
    "adapter": ["adapter", "img", "image", "image_features", "features", "feat", "z"],
    "fused": ["fused", "fusion", "img", "image_features_main", "image_features", "features", "feat", "z"],
}

# 可视化采样，避免 UMAP/t-SNE/heatmap 过大。
MAX_ID_PER_CLASS_FOR_FEATURE = 80
MAX_OOD_FOR_FEATURE = 800
SIM_MATRIX_ID_PER_CLASS = 50
SIM_MATRIX_OOD_TOTAL = 500
RANDOM_STATE = 2026

# 输出目录
ANALYSIS_ROOT = OUTPUT_ROOT / "analysis" / "bayesrt_vs_bayesadapter_msp_uncertainty_rc_analysis"
CACHE_ROOT = ANALYSIS_ROOT / "caches"
PRED_CACHE_ROOT = CACHE_ROOT / "predictions"
FEATURE_CACHE_ROOT = CACHE_ROOT / "features"
UNCERTAINTY_CACHE_ROOT = CACHE_ROOT / "uncertainty"
SUMMARY_ROOT = ANALYSIS_ROOT / "summaries"
FIGURE_ROOT = ANALYSIS_ROOT / "figures"
PAPER_READY_ROOT = ANALYSIS_ROOT / "paper_ready"

FIG_DIRS = {
    "msp_kde": FIGURE_ROOT / "msp_uncertainty_kde",
    "msp_ecdf": FIGURE_ROOT / "msp_uncertainty_ecdf",
    "msp_cwo": FIGURE_ROOT / "msp_correct_wrong_ood",
    "feature_embedding": FIGURE_ROOT / "feature_embedding_R_C_adapter",
    "similarity_matrix": FIGURE_ROOT / "similarity_matrix_R_C_adapter",
    "similarity_distribution": FIGURE_ROOT / "similarity_distribution_R_C_adapter",
    "calibration": FIGURE_ROOT / "calibration",
    "risk_coverage": FIGURE_ROOT / "risk_coverage",
    "metric_delta": FIGURE_ROOT / "metric_delta_msp",
}

for p in [CACHE_ROOT, PRED_CACHE_ROOT, FEATURE_CACHE_ROOT, UNCERTAINTY_CACHE_ROOT, SUMMARY_ROOT, FIGURE_ROOT, PAPER_READY_ROOT, *FIG_DIRS.values()]:
    p.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_VERSION:", NOTEBOOK_VERSION)
print("REPO_ROOT:", REPO_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("ANALYSIS_ROOT:", ANALYSIS_ROOT)
print("MAIN_UNCERTAINTY_SCORE:", MAIN_UNCERTAINTY_SCORE)


NOTEBOOK_VERSION: msp_uncertainty_rc_analysis_v4_checked_2026_05_24
REPO_ROOT: /root/autodl-tmp/MMRL
DATASET_ROOT: /root/autodl-tmp/MMRL/DATASETS
OUTPUT_ROOT: /root/autodl-tmp/MMRL/output_refactor
ANALYSIS_ROOT: /root/autodl-tmp/MMRL/output_refactor/analysis/bayesrt_vs_bayesadapter_msp_uncertainty_rc_analysis
MAIN_UNCERTAINTY_SCORE: msp_uncertainty


In [2]:

# =========================
# 1. 环境初始化
# =========================
import os
import sys
import json
import math
import importlib
import warnings
from argparse import Namespace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("TORCH_NUM_THREADS", "1")
os.environ.setdefault("TORCH_NUM_INTEROP_THREADS", "1")

REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dassl.engine import build_trainer
from dassl.utils import set_random_seed, setup_logger

from core.config import setup_cfg
from core.utils import import_optional_modules
from eval_ood import build_ood_loader

# sklearn 用于 AUROC/AUPR/calibration/risk-coverage。
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
)

try:
    from scipy.stats import gaussian_kde
    SCIPY_AVAILABLE = True
except Exception:
    gaussian_kde = None
    SCIPY_AVAILABLE = False

try:
    from sklearn.manifold import TSNE
    from sklearn.decomposition import PCA
    SKLEARN_EMBED_AVAILABLE = True
except Exception:
    TSNE = None
    PCA = None
    SKLEARN_EMBED_AVAILABLE = False

try:
    import umap
    UMAP_AVAILABLE = True
except Exception:
    umap = None
    UMAP_AVAILABLE = False


def import_runtime_modules():
    import_optional_modules([
        "datasets.cifar_10",
        "datasets.ood_image_datasets",
        "datasets.oxford_pets",
        "datasets.oxford_flowers",
        "datasets.fgvc_aircraft",
        "datasets.dtd",
        "datasets.eurosat",
        "datasets.stanford_cars",
        "datasets.food101",
        "datasets.sun397",
        "datasets.caltech101",
        "datasets.ucf101",
        "datasets.imagenet",
        "datasets.imagenetv2",
        "datasets.imagenet_sketch",
        "datasets.imagenet_a",
        "datasets.imagenet_r",
    ])
    importlib.import_module("trainers.refactor_runner")

import_runtime_modules()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

try:
    import subprocess
    git_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT).decode().strip()
except Exception as e:
    git_head = f"unknown: {e}"
print("git_head =", git_head)
print("SCIPY_AVAILABLE =", SCIPY_AVAILABLE)
print("UMAP_AVAILABLE =", UMAP_AVAILABLE)
print("SKLEARN_EMBED_AVAILABLE =", SKLEARN_EMBED_AVAILABLE)

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)
warnings.filterwarnings("ignore", category=UserWarning)

device = cuda
git_head = f47857efdf39da4c9c0185a30518448e28b0e126
SCIPY_AVAILABLE = True
UMAP_AVAILABLE = False
SKLEARN_EMBED_AVAILABLE = True


## 2. 方法配置、checkpoint 路径与 cache 路径

路径规则与原 notebook 保持一致：

- `BayesRTMMRL`：`output_refactor/BayesRTMMRL/FS/fewshot_train/<dataset>/shots_<shot>/ViT-B-16/default/seed<seed>`
- `BayesAdapter`：`output_refactor/ClipAdapters/BAYES_ADAPTER/FS/fewshot_train/<dataset>/shots_<shot>/ViT-B-16/seed<seed>`

In [3]:

# =========================
# 2. 方法配置与路径规则
# =========================
METHOD_SPECS = {
    "BayesRTMMRL": {
        "requested_method": "BayesRTMMRL",
        "launch_method": "BayesRTMMRL",
        "method_config_file": "configs/methods/bayesrt_mmrl.yaml",
        "runtime_config_file": "configs/runtime/mmrl_family.yaml",
        "run_tag": "default",
        "is_adapter": False,
    },
    "BayesAdapter": {
        "requested_method": "BayesAdapter",
        "launch_method": "ClipAdapters",
        "method_config_file": "configs/methods/clip_adapters_bayes.yaml",
        "runtime_config_file": "configs/runtime/adapter_family.yaml",
        "run_tag": "BAYES_ADAPTER",
        "is_adapter": True,
    },
}


def protocol_phase(protocol: str):
    protocol = str(protocol).upper()
    if protocol == "FS":
        return "fewshot_train", "all"
    if protocol == "B2N":
        return "train_base", "base"
    if protocol == "CD":
        return "cross_train", "all"
    raise ValueError(f"Unsupported protocol: {protocol}")


def dataset_config_file(dataset: str) -> Path:
    return REPO_ROOT / "configs" / "datasets" / f"{dataset}.yaml"


def protocol_config_file(protocol: str) -> Path:
    protocol = str(protocol).upper()
    if protocol == "FS":
        return REPO_ROOT / "configs" / "protocols" / "fs.yaml"
    if protocol == "B2N":
        return REPO_ROOT / "configs" / "protocols" / "b2n.yaml"
    if protocol == "CD":
        return REPO_ROOT / "configs" / "protocols" / "cd.yaml"
    raise ValueError(f"Unsupported protocol: {protocol}")


def backbone_dir(backbone: str) -> str:
    return str(backbone).replace("/", "-")


def build_model_dir(method_name: str, dataset: str, shot: int, seed: int) -> Path:
    spec = METHOD_SPECS[method_name]
    phase, _subsample = protocol_phase(PROTOCOL)
    bdir = backbone_dir(BACKBONE)
    if spec["is_adapter"]:
        return OUTPUT_ROOT / spec["launch_method"] / spec["run_tag"] / PROTOCOL / phase / dataset / f"shots_{shot}" / bdir / f"seed{seed}"
    return OUTPUT_ROOT / spec["launch_method"] / PROTOCOL / phase / dataset / f"shots_{shot}" / bdir / spec["run_tag"] / f"seed{seed}"


def build_case_cache_dir(method_name: str, id_dataset: str, shot: int, seed: int) -> Path:
    return PRED_CACHE_ROOT / method_name / PROTOCOL / id_dataset / f"shots_{shot}" / backbone_dir(BACKBONE) / f"seed{seed}"


def build_feature_cache_dir(method_name: str, feature_branch: str, id_dataset: str, shot: int, seed: int) -> Path:
    return FEATURE_CACHE_ROOT / method_name / feature_branch / PROTOCOL / id_dataset / f"shots_{shot}" / backbone_dir(BACKBONE) / f"seed{seed}"

for m in METHOD_SPECS:
    print(m, METHOD_SPECS[m])

BayesRTMMRL {'requested_method': 'BayesRTMMRL', 'launch_method': 'BayesRTMMRL', 'method_config_file': 'configs/methods/bayesrt_mmrl.yaml', 'runtime_config_file': 'configs/runtime/mmrl_family.yaml', 'run_tag': 'default', 'is_adapter': False}
BayesAdapter {'requested_method': 'BayesAdapter', 'launch_method': 'ClipAdapters', 'method_config_file': 'configs/methods/clip_adapters_bayes.yaml', 'runtime_config_file': 'configs/runtime/adapter_family.yaml', 'run_tag': 'BAYES_ADAPTER', 'is_adapter': True}


## 3. 构建 trainer 与加载 checkpoint

此部分不训练，只构建模型并加载已有 checkpoint。缺失 checkpoint 会记录到 summary，不会中断整个批处理。

In [4]:

# =========================
# 3. 构建 trainer
# =========================
def make_args(method_name: str, id_dataset: str, shot: int, seed: int, model_dir: Path, output_dir: Path):
    spec = METHOD_SPECS[method_name]
    _phase, subsample = protocol_phase(PROTOCOL)
    exec_mode = EXEC_MODE_BY_METHOD.get(method_name, "online")

    dcfg = dataset_config_file(id_dataset)
    pcfg = protocol_config_file(PROTOCOL)
    mcfg = REPO_ROOT / spec["method_config_file"]
    rcfg = REPO_ROOT / spec["runtime_config_file"]

    for path in [dcfg, pcfg, mcfg, rcfg]:
        if not path.exists():
            raise FileNotFoundError(path)

    opts = [
        "DATASET.NUM_SHOTS", str(shot),
        "DATASET.SUBSAMPLE_CLASSES", subsample,
        "MODEL.BACKBONE.NAME", BACKBONE,
    ]

    return Namespace(
        root=str(DATASET_ROOT),
        output_dir=str(output_dir),
        dataset_config_file=str(dcfg),
        method_config_file=str(mcfg),
        protocol_config_file=str(pcfg),
        runtime_config_file=str(rcfg),
        exp_config="",
        method=spec["launch_method"],
        protocol=PROTOCOL,
        exec_mode=exec_mode,
        seed=int(seed),
        trainer="RefactorRunner",
        eval_only=True,
        model_dir=str(model_dir),
        load_epoch=LOAD_EPOCH,
        no_train=True,
        opts=opts,
    )


def build_loaded_trainer(method_name: str, id_dataset: str, shot: int, seed: int):
    model_dir = build_model_dir(method_name, id_dataset, shot, seed)
    cache_dir = build_case_cache_dir(method_name, id_dataset, shot, seed)
    cache_dir.mkdir(parents=True, exist_ok=True)

    if not model_dir.exists():
        raise FileNotFoundError(f"checkpoint dir not found: {model_dir}")

    args = make_args(
        method_name=method_name,
        id_dataset=id_dataset,
        shot=shot,
        seed=seed,
        model_dir=model_dir,
        output_dir=cache_dir / "_runtime_output",
    )

    if int(seed) >= 0:
        set_random_seed(int(seed))

    cfg = setup_cfg(args)
    setup_logger(cfg.OUTPUT_DIR)

    trainer = build_trainer(cfg)
    trainer.load_model(str(model_dir), epoch=LOAD_EPOCH)
    trainer.set_model_mode("eval")
    return trainer, model_dir, cache_dir

## 4. MSP uncertainty 与基础指标函数

主分数统一为：

```text
MSP confidence = max_c softmax(logits)_c
MSP uncertainty = 1 - MSP confidence
```

OOD detection 中分数方向固定为：

```text
score 越大 = 越不确定 = 越倾向 OOD / 错误样本
```

In [5]:

# =========================
# 4. MSP uncertainty 与指标函数
# =========================
def softmax_numpy(logits):
    x = np.asarray(logits, dtype=np.float64)
    x = x - np.max(x, axis=1, keepdims=True)
    e = np.exp(x)
    return e / np.clip(e.sum(axis=1, keepdims=True), 1e-12, None)


def probs_from_logits_tensor(logits_t: torch.Tensor) -> torch.Tensor:
    return F.softmax(logits_t.float(), dim=1)


def msp_confidence_from_logits(logits_t: torch.Tensor) -> torch.Tensor:
    return probs_from_logits_tensor(logits_t).max(dim=1).values.detach().float().cpu()


def msp_uncertainty_from_logits(logits_t: torch.Tensor) -> torch.Tensor:
    return (1.0 - msp_confidence_from_logits(logits_t)).detach().float().cpu()


def predictive_entropy_from_logits(logits_t: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    p = probs_from_logits_tensor(logits_t).clamp_min(eps)
    return (-(p * p.log()).sum(dim=1)).detach().float().cpu()


def top1_pred_from_logits(logits_t: torch.Tensor) -> torch.Tensor:
    return logits_t.float().argmax(dim=1).detach().long().cpu()


def fpr_at_tpr(y_true, scores, target_tpr=0.95):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)
    order = np.argsort(-scores)
    y = y_true[order]
    pos = max(int((y == 1).sum()), 1)
    neg = max(int((y == 0).sum()), 1)
    tp = np.cumsum(y == 1)
    fp = np.cumsum(y == 0)
    tpr = tp / pos
    fpr = fp / neg
    idx = np.where(tpr >= target_tpr)[0]
    if len(idx) == 0:
        return np.nan
    return float(np.min(fpr[idx]))


def detection_error(y_true, scores):
    """Minimum detection error with equal class prior; positive class has larger score."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)
    thresholds = np.r_[np.inf, np.sort(np.unique(scores))[::-1], -np.inf]
    pos = max(int((y_true == 1).sum()), 1)
    neg = max(int((y_true == 0).sum()), 1)
    best = np.inf
    for t in thresholds:
        pred_pos = scores >= t
        fnr = ((~pred_pos) & (y_true == 1)).sum() / pos
        fpr = (pred_pos & (y_true == 0)).sum() / neg
        best = min(best, 0.5 * (fnr + fpr))
    return float(best)


def compute_ood_metrics_from_uncertainty(id_uncertainty, ood_uncertainty):
    """ID=0, OOD=1, larger score means more likely OOD."""
    id_s = np.asarray(id_uncertainty, dtype=float)
    ood_s = np.asarray(ood_uncertainty, dtype=float)
    y_true = np.r_[np.zeros_like(id_s, dtype=int), np.ones_like(ood_s, dtype=int)]
    scores = np.r_[id_s, ood_s]

    if len(np.unique(y_true)) < 2 or len(np.unique(scores)) < 2:
        auroc = np.nan
    else:
        auroc = float(roc_auc_score(y_true, scores))

    return {
        "AUROC": auroc,
        "AUPR_OUT": float(average_precision_score(y_true, scores)),
        "AUPR_IN": float(average_precision_score(1 - y_true, -scores)),
        "FPR95": fpr_at_tpr(y_true, scores, target_tpr=0.95),
        "DetectionError": detection_error(y_true, scores),
    }


def compute_error_detection_metrics(correct_bool, uncertainty):
    """ID 内部错误检测：positive=incorrect, score=uncertainty."""
    correct_bool = np.asarray(correct_bool).astype(bool)
    scores = np.asarray(uncertainty, dtype=float)
    y_true = (~correct_bool).astype(int)
    if len(np.unique(y_true)) < 2 or len(np.unique(scores)) < 2:
        return {"ErrorAUROC": np.nan, "ErrorAUPR": np.nan, "ErrorFPR95": np.nan}
    return {
        "ErrorAUROC": float(roc_auc_score(y_true, scores)),
        "ErrorAUPR": float(average_precision_score(y_true, scores)),
        "ErrorFPR95": fpr_at_tpr(y_true, scores, target_tpr=0.95),
    }

## 5. R/C 分支特征抽取工具

`BayesRTMMRL` 必须保存 R 分支与 C 分支特征。由于项目实现中 `outputs.features` 的 key 可能不同，这里用候选 key 列表做鲁棒匹配。

如果运行时某个分支没有匹配到特征，notebook 会在 `feature_key_report.csv` 里记录，方便你根据真实 key 修改 `FEATURE_KEY_CANDIDATES`。

In [6]:

# =========================
# 5. R/C 分支特征抽取工具
# =========================
def as_cpu_float_tensor(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().float().cpu()
    try:
        return torch.as_tensor(x).detach().float().cpu()
    except Exception:
        return None


def get_attr_or_key(obj, key, default=None):
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def extract_feature_dict_from_outputs(outputs):
    """Return a dictionary-like feature container from method outputs."""
    candidates = []
    for name in ["features", "feature", "feats", "embeddings", "embedding"]:
        val = get_attr_or_key(outputs, name, None)
        if val is not None:
            candidates.append(val)
    # Some implementations return a dict directly.
    if isinstance(outputs, dict):
        candidates.append(outputs)

    for val in candidates:
        if isinstance(val, dict):
            return val
    return {}


def pick_feature_from_dict(feature_dict, branch_name):
    candidates = FEATURE_KEY_CANDIDATES.get(branch_name, [branch_name])
    available = list(feature_dict.keys()) if isinstance(feature_dict, dict) else []

    # Exact candidates first.
    for k in candidates:
        if isinstance(feature_dict, dict) and k in feature_dict:
            return as_cpu_float_tensor(feature_dict[k]), k, available

    # Case-insensitive exact match.
    lowered = {str(k).lower(): k for k in available}
    for k in candidates:
        lk = str(k).lower()
        if lk in lowered:
            real_k = lowered[lk]
            return as_cpu_float_tensor(feature_dict[real_k]), real_k, available

    # Substring fallback for R/C/adapter.
    for real_k in available:
        rk = str(real_k).lower()
        for cand in candidates:
            if str(cand).lower() in rk:
                return as_cpu_float_tensor(feature_dict[real_k]), real_k, available

    return None, None, available


def expected_branches_for_method(method_name):
    return FEATURE_BRANCHES.get(method_name, ["adapter"])


def normalize_feature_tensor_shape(feat):
    """Ensure [N, D]. If [N, ...], flatten all dimensions except batch."""
    if feat is None:
        return None
    if feat.ndim == 1:
        return feat[:, None]
    if feat.ndim > 2:
        return feat.flatten(start_dim=1)
    return feat


def l2_normalize_np(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float64)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(n, eps, None)


def bayesrt_eval_use_posterior_mean(method, eval_ctx):
    """Mirror BayesRTMMRLMethod.forward_eval posterior-mean policy.

    The current project exposes C branch in MethodOutputs.features["img"], but
    it does not expose the R branch there. R/C image features are available in
    BayesRTMMRLModel.forward_joint as:
        - image_features_rep  -> R branch
        - image_features_main -> C branch
    This helper keeps feature extraction aligned with the method's eval policy.
    """
    use_posterior_mean = bool(getattr(method, "eval_use_posterior_mean", False))
    try:
        cfg = method.cfg.BAYESRT_MMRL
        if bool(getattr(cfg, "NOVEL_TEXT_MEAN_ONLY", False)):
            protocol = str(eval_ctx.protocol).upper()
            dataset = str(eval_ctx.dataset_name)
            sub_cls = str(eval_ctx.subsample_classes or "all")
            is_b2n_novel = protocol == "B2N" and sub_cls != "base"
            is_cd_target = protocol == "CD" and dataset != "ImageNet"
            if is_b2n_novel or is_cd_target:
                use_posterior_mean = True
    except Exception:
        pass
    return use_posterior_mean


@torch.no_grad()
def extract_bayesrt_rc_features(method, batch, eval_ctx):
    """Directly extract BayesRTMMRL R/C image features from forward_joint.

    This is required for this repository revision because BayesRTMMRLMethod.forward_eval
    returns features={"img": image_features_main, "text": text_features}; it does
    not include image_features_rep. Without this direct path the notebook cannot
    produce the requested R-branch plots or similarity matrices.
    """
    if not hasattr(method, "model") or not hasattr(method.model, "forward_joint"):
        return {}, [], "missing_forward_joint"
    if not isinstance(batch, dict) or "img" not in batch:
        return {}, [], "missing_img_batch"

    image = batch["img"].to(method.device)
    num_samples = int(max(1, getattr(method, "n_mc_test", 1)))
    use_posterior_mean = bayesrt_eval_use_posterior_mean(method, eval_ctx)

    out = method.model.forward_joint(
        image=image,
        num_samples=num_samples,
        use_posterior_mean=use_posterior_mean,
    )
    available = sorted([str(k) for k in out.keys()]) if isinstance(out, dict) else []
    if not isinstance(out, dict):
        return {}, available, "forward_joint_non_dict"

    feats = {
        "R": as_cpu_float_tensor(out.get("image_features_rep")),
        "C": as_cpu_float_tensor(out.get("image_features_main")),
    }
    return feats, available, "forward_joint"


def add_feature_chunk(features_by_branch, feature_key_report, branch, feat, matched_key, available_keys):
    feat = normalize_feature_tensor_shape(feat)
    if feat is not None:
        features_by_branch[branch].append(feat)
    if matched_key is not None:
        feature_key_report[branch]["matched_keys"].append(str(matched_key))
    feature_key_report[branch]["available_keys_examples"].append([str(x) for x in available_keys])


## 6. Forward 收集：logits、MSP uncertainty、labels、correct、features

保存 payload 的核心字段：

```python
{
    "logits": Tensor[N, C],
    "labels": Tensor[N] or None,
    "preds": Tensor[N],
    "msp_confidence": Tensor[N],
    "msp_uncertainty": Tensor[N],
    "predictive_entropy": Tensor[N],
    "correct": Tensor[N] or None,
    "features": {"R": Tensor[N,D], "C": Tensor[N,D], ...},
    "feature_key_report": {...}
}
```

In [7]:

# =========================
# 6. collector
# =========================
@torch.no_grad()
def collect_selected_outputs(trainer, loader, split_name: str, method_name: str, save_features: bool = True):
    eval_ctx = trainer.executor.build_eval_context(trainer, split_name)
    method = trainer.method
    method.eval()
    trainer.set_model_mode("eval")

    logits_all = []
    labels_all = []
    branches = expected_branches_for_method(method_name)
    features_by_branch = {b: [] for b in branches}
    feature_key_report = {b: {"matched_keys": [], "available_keys_examples": []} for b in branches}

    use_amp = False
    try:
        prec = method.get_precision()
        use_amp = (str(prec).lower() == "amp" and torch.cuda.is_available())
    except Exception:
        use_amp = torch.cuda.is_available()

    for batch_idx, batch in enumerate(loader):
        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = method.forward_eval(batch, eval_ctx)
            logits = method.select_eval_logits(outputs, eval_ctx)

        logits_all.append(logits.detach().float().cpu())

        labels = getattr(outputs, "labels", None)
        if labels is None and isinstance(outputs, dict):
            labels = outputs.get("labels", None)
        if labels is None and isinstance(batch, dict) and "label" in batch:
            labels = batch["label"]
        if labels is not None:
            labels_all.append(labels.detach().long().cpu())

        if save_features:
            # Repository-specific BayesRTMMRL path:
            # forward_eval exposes only C/main feature as outputs.features["img"],
            # while R/rep is only available from model.forward_joint.
            if method_name == "BayesRTMMRL":
                rc_feats, available_keys, source = extract_bayesrt_rc_features(method, batch, eval_ctx)
                for branch in branches:
                    feat = rc_feats.get(branch)
                    if feat is not None:
                        matched_key = "forward_joint.image_features_rep" if branch == "R" else "forward_joint.image_features_main"
                        feat = normalize_feature_tensor_shape(feat)
                        if feat is not None:
                            features_by_branch[branch].append(feat)
                        feature_key_report[branch]["matched_keys"].append(matched_key)
                        if batch_idx < 3:
                            feature_key_report[branch]["available_keys_examples"].append(available_keys)
                    else:
                        # Fallback: try MethodOutputs.features for C/main in case direct path fails.
                        feat_dict = extract_feature_dict_from_outputs(outputs)
                        feat2, matched_key2, available2 = pick_feature_from_dict(feat_dict, branch)
                        feat2 = normalize_feature_tensor_shape(feat2)
                        if feat2 is not None:
                            features_by_branch[branch].append(feat2)
                        if matched_key2 is not None:
                            feature_key_report[branch]["matched_keys"].append(str(matched_key2))
                        if batch_idx < 3:
                            feature_key_report[branch]["available_keys_examples"].append([str(x) for x in available2] + [f"bayesrt_direct_status={source}"])
            else:
                feat_dict = extract_feature_dict_from_outputs(outputs)
                for branch in branches:
                    feat, matched_key, available_keys = pick_feature_from_dict(feat_dict, branch)
                    feat = normalize_feature_tensor_shape(feat)
                    if feat is not None:
                        features_by_branch[branch].append(feat)
                    if matched_key is not None:
                        feature_key_report[branch]["matched_keys"].append(str(matched_key))
                    if batch_idx < 3:
                        feature_key_report[branch]["available_keys_examples"].append([str(x) for x in available_keys])

    logits = torch.cat(logits_all, dim=0)
    labels = torch.cat(labels_all, dim=0) if labels_all else None
    preds = top1_pred_from_logits(logits)
    msp_conf = msp_confidence_from_logits(logits)
    msp_unc = 1.0 - msp_conf
    pred_entropy = predictive_entropy_from_logits(logits)
    correct = (preds == labels).detach().cpu() if labels is not None else None

    features = {}
    for branch, chunks in features_by_branch.items():
        if chunks:
            try:
                features[branch] = torch.cat(chunks, dim=0)
            except Exception as e:
                print(f"[WARN] failed to concat features: method={method_name} branch={branch} error={repr(e)}")

    # compress report
    for branch, rep in feature_key_report.items():
        rep["matched_keys"] = sorted(set(rep["matched_keys"]))
        # avoid huge CSV cells while keeping early-batch diagnostics
        rep["available_keys_examples"] = rep["available_keys_examples"][:3]

    return {
        "logits": logits,
        "labels": labels,
        "preds": preds,
        "msp_confidence": msp_conf,
        "msp_uncertainty": msp_unc.detach().float().cpu(),
        "predictive_entropy": pred_entropy,
        "correct": correct,
        "features": features,
        "feature_key_report": feature_key_report,
    }


def save_tensor_payload(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)


def load_tensor_payload(path: Path):
    return torch.load(path, map_location="cpu")


def payload_meta(method_name, id_dataset, shot, seed, split, ood_dataset=None, model_dir=None):
    return {
        "notebook_version": NOTEBOOK_VERSION,
        "method": method_name,
        "id_dataset": id_dataset,
        "ood_dataset": ood_dataset,
        "protocol": PROTOCOL,
        "shot": int(shot),
        "seed": int(seed),
        "backbone": BACKBONE,
        "split": split,
        "model_dir": str(model_dir) if model_dir is not None else "",
        "main_uncertainty_score": MAIN_UNCERTAINTY_SCORE,
        "save_features": bool(SAVE_FEATURES),
        "feature_branches": expected_branches_for_method(method_name),
        "git_head": git_head,
    }


## 7. 单个 method × ID dataset × shot × seed 的执行逻辑

每个 case 先跑一次 ID test set，再依次跑多个 OOD dataset。ID 结果会被复用。

In [8]:

# =========================
# 7. run one case
# =========================
def run_one_case(method_name: str, id_dataset: str, shot: int, seed: int):
    model_dir = build_model_dir(method_name, id_dataset, shot, seed)
    cache_dir = build_case_cache_dir(method_name, id_dataset, shot, seed)
    cache_dir.mkdir(parents=True, exist_ok=True)

    result_csv = cache_dir / "ood_msp_results.csv"
    status_json = cache_dir / "status.json"

    if (not RECOMPUTE_CACHE) and result_csv.exists():
        print(f"[SKIP] {method_name} {id_dataset} shot={shot} seed={seed}: cached {result_csv}")
        return pd.read_csv(result_csv)

    try:
        trainer, model_dir, cache_dir = build_loaded_trainer(method_name, id_dataset, shot, seed)
    except Exception as e:
        row = {
            "method": method_name,
            "id_dataset": id_dataset,
            "shot": int(shot),
            "seed": int(seed),
            "status": "missing_or_failed_build",
            "error": repr(e),
            "model_dir": str(model_dir),
        }
        with status_json.open("w", encoding="utf-8") as f:
            json.dump(row, f, indent=2, ensure_ascii=False)
        print("[FAIL_BUILD]", row)
        return pd.DataFrame([row])

    print(f"[ID] collect outputs: {method_name} {id_dataset} shot={shot} seed={seed}")
    id_payload_path = cache_dir / "id_test_outputs.pt"
    id_payload = collect_selected_outputs(
        trainer=trainer,
        loader=trainer.test_loader,
        split_name="test",
        method_name=method_name,
        save_features=SAVE_FEATURES,
    )
    id_payload.update({
        "method": method_name,
        "split": "id_test",
        "meta": payload_meta(method_name, id_dataset, shot, seed, "id_test", model_dir=model_dir),
    })
    save_tensor_payload(id_payload_path, id_payload)

    id_unc = id_payload["msp_uncertainty"].detach().cpu().numpy()
    id_conf = id_payload["msp_confidence"].detach().cpu().numpy()
    id_correct = id_payload["correct"].detach().cpu().numpy() if id_payload.get("correct", None) is not None else None

    rows = []
    feature_report_rows = []
    for branch, rep in id_payload.get("feature_key_report", {}).items():
        feature_report_rows.append({
            "method": method_name,
            "id_dataset": id_dataset,
            "ood_dataset": "",
            "shot": int(shot),
            "seed": int(seed),
            "split": "id_test",
            "feature_branch": branch,
            "matched_keys": ";".join(rep.get("matched_keys", [])),
            "available_keys_examples": json.dumps(rep.get("available_keys_examples", []), ensure_ascii=False),
        })

    for ood_dataset in OOD_DATASETS:
        print(f"[OOD] collect outputs: {method_name} ID={id_dataset} OOD={ood_dataset} shot={shot} seed={seed}")
        try:
            registry_name, ood_loader = build_ood_loader(
                cfg=trainer.cfg,
                ood_dataset_key=ood_dataset,
                batch_size=int(OOD_BATCH_SIZE),
                num_workers=int(OOD_NUM_WORKERS),
            )

            ood_payload = collect_selected_outputs(
                trainer=trainer,
                loader=ood_loader,
                split_name=f"ood_{ood_dataset}",
                method_name=method_name,
                save_features=SAVE_FEATURES,
            )
            ood_payload.update({
                "method": method_name,
                "split": "ood",
                "meta": payload_meta(method_name, id_dataset, shot, seed, "ood", ood_dataset=ood_dataset, model_dir=model_dir),
            })

            ood_payload_path = cache_dir / f"ood_{ood_dataset}_outputs.pt"
            save_tensor_payload(ood_payload_path, ood_payload)

            ood_unc = ood_payload["msp_uncertainty"].detach().cpu().numpy()
            metrics = compute_ood_metrics_from_uncertainty(id_unc, ood_unc)

            row = {
                "method": method_name,
                "id_dataset": id_dataset,
                "ood_dataset": ood_dataset,
                "registry_dataset": registry_name,
                "shot": int(shot),
                "seed": int(seed),
                "score_name": "msp_uncertainty",
                "num_id": int(len(id_unc)),
                "num_ood": int(len(ood_unc)),
                "id_msp_uncertainty_mean": float(np.mean(id_unc)),
                "ood_msp_uncertainty_mean": float(np.mean(ood_unc)),
                "id_msp_confidence_mean": float(np.mean(id_conf)),
                "status": "ok",
                "id_payload_path": str(id_payload_path),
                "ood_payload_path": str(ood_payload_path),
                **metrics,
            }
            if id_correct is not None:
                row.update(compute_error_detection_metrics(id_correct, id_unc))

            for branch, rep in ood_payload.get("feature_key_report", {}).items():
                feature_report_rows.append({
                    "method": method_name,
                    "id_dataset": id_dataset,
                    "ood_dataset": ood_dataset,
                    "shot": int(shot),
                    "seed": int(seed),
                    "split": "ood",
                    "feature_branch": branch,
                    "matched_keys": ";".join(rep.get("matched_keys", [])),
                    "available_keys_examples": json.dumps(rep.get("available_keys_examples", []), ensure_ascii=False),
                })

        except Exception as e:
            row = {
                "method": method_name,
                "id_dataset": id_dataset,
                "ood_dataset": ood_dataset,
                "registry_dataset": "",
                "shot": int(shot),
                "seed": int(seed),
                "score_name": "msp_uncertainty",
                "num_id": int(len(id_unc)),
                "num_ood": 0,
                "status": "failed_ood",
                "error": repr(e),
                "id_payload_path": str(id_payload_path),
                "ood_payload_path": "",
            }
            print("[FAIL_OOD]", row)

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(result_csv, index=False)

    if feature_report_rows:
        pd.DataFrame(feature_report_rows).to_csv(cache_dir / "feature_key_report.csv", index=False)

    with status_json.open("w", encoding="utf-8") as f:
        json.dump({
            "method": method_name,
            "id_dataset": id_dataset,
            "shot": int(shot),
            "seed": int(seed),
            "status": "done",
            "result_csv": str(result_csv),
            "model_dir": str(model_dir),
        }, f, indent=2, ensure_ascii=False)

    return df

## 8. 批量执行并保存 raw OOD 结果

建议先把 `SHOTS=[16]`、`SEEDS=[1]` 小范围验证，再跑全量。

In [9]:

# =========================
# 8. 批量执行
# =========================
all_rows = []

for id_dataset in ID_DATASETS:
    for shot in SHOTS:
        for seed in SEEDS:
            for method_name in ["BayesRTMMRL", "BayesAdapter"]:
                print("=" * 100)
                print(f"RUN method={method_name} id={id_dataset} shot={shot} seed={seed}")
                df_case = run_one_case(method_name, id_dataset, shot, seed)
                all_rows.append(df_case)

ood_raw = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
raw_csv = SUMMARY_ROOT / "summary_ood_msp_raw.csv"
ood_raw.to_csv(raw_csv, index=False)

print("Saved raw summary:", raw_csv)
display(ood_raw.head(30))

RUN method=BayesRTMMRL id=cifar_10 shot=8 seed=1
Loading trainer: RefactorRunner
Loading dataset: CIFAR_10
Loading preprocessed few-shot data from /root/autodl-tmp/MMRL/DATASETS/cifar10/split_fewshot/shot_8-seed_1.pkl
Building transform_train
+ random resized crop (size=(224, 224), scale=(0.5, 1))
+ random flip
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
---------  --------
Dataset    CIFAR_10
# classes  10
# train_x  80
# val      40
# test     10,000
---------  --------
[BayesRTMMRL] trainable params: {'representation_learner.compound_rep_tokens_r2tproj.3.weight', 'representation_learner.compound_rep_tokens_r2tproj.4.weight', 'text_posterior.posterior_rho', 'representation_learner.com

/root/autodl-tmp/MMRL/trainers/refactor_runner.py:73: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if prec == "amp" else None
/tmp/ipykernel_12854/4283529764.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[OOD] collect outputs: BayesRTMMRL ID=cifar_10 OOD=dtd shot=8 seed=1
Reading split from /root/autodl-tmp/MMRL/DATASETS/dtd/split_zhou_DescribableTextures.json
[OOD] dtd: registry=DescribableTextures, num_test=1692
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
[OOD] collect outputs: BayesRTMMRL ID=cifar_10 OOD=tinyimagenet shot=8 seed=1
[OODImageFolder] TinyImageNet: loaded 10000 images from /root/autodl-tmp/MMRL/DATASETS/tiny-imagenet-200/test/images
[OOD] tinyimagenet: registry=TinyImageNetOOD, num_test=10000
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])


/tmp/ipykernel_12854/4283529764.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[OOD] collect outputs: BayesRTMMRL ID=cifar_10 OOD=oxford_flowers shot=8 seed=1
Reading split from /root/autodl-tmp/MMRL/DATASETS/oxford_flowers/split_zhou_OxfordFlowers.json
[OOD] oxford_flowers: registry=OxfordFlowers, num_test=2463
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])


...

[OOD] collect outputs: BayesAdapter ID=cifar_10 OOD=sun397 shot=8 seed=1
Reading split from /root/autodl-tmp/MMRL/DATASETS/sun397/split_zhou_SUN397.json
[OOD] sun397: registry=SUN397, num_test=19850
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])


/tmp/ipykernel_12854/4283529764.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Saved raw summary: /root/autodl-tmp/MMRL/output_refactor/analysis/bayesrt_vs_bayesadapter_msp_uncertainty_rc_analysis/summaries/summary_ood_msp_raw.csv


,method,id_dataset,ood_dataset,registry_dataset,shot,seed,score_name,num_id,num_ood,id_msp_uncertainty_mean,ood_msp_uncertainty_mean,id_msp_confidence_mean,status,id_payload_path,ood_payload_path,AUROC,AUPR_OUT,AUPR_IN,FPR95,DetectionError,ErrorAUROC,ErrorAUPR,ErrorFPR95
0,BayesRTMMRL,cifar_10,dtd,DescribableTextures,8,1,msp_uncertainty,10000,1692,0.052760,0.540974,0.947240,ok,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,0.971709,0.864790,0.994594,0.1156,0.079592,0.907107,0.380174,0.313013
1,BayesRTMMRL,cifar_10,tinyimagenet,TinyImageNetOOD,8,1,msp_uncertainty,10000,10000,0.052760,0.407457,0.947240,ok,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,0.905695,0.911133,0.886796,0.4643,0.161950,0.907107,0.380174,0.313013
2,BayesRTMMRL,cifar_10,oxford_flowers,OxfordFlowers,8,1,msp_uncertainty,10000,2463,0.052760,0.552039,0.947240,ok,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,0.978593,0.909995,0.994888,0.0963,0.072252,0.907107,0.380174,0.313013
3,BayesRTMMRL,cifar_10,sun397,SUN397,8,1,msp_uncertainty,10000,19850,0.052760,0.442971,0.947240,ok,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,0.936290,0.964003,0.877175,0.2608,0.127326,0.907107,0.380174,0.313013
4,BayesAdapter,cifar_10,dtd,DescribableTextures,8,1,msp_uncertainty,10000,1692,0.133499,0.609880,0.866501,ok,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,0.947682,0.719313,0.990322,0.2038,0.118150,0.911740,0.462958,0.304837
5,BayesAdapter,cifar_10,tinyimagenet,TinyImageNetOOD,8,1,msp_uncertainty,10000,10000,0.133499,0.497639,0.866501,ok,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,0.860875,0.861882,0.812059,0.7105,0.197400,0.911740,0.462958,0.304837
6,BayesAdapter,cifar_10,oxford_flowers,OxfordFlowers,8,1,msp_uncertainty,10000,2463,0.133499,0.586156,0.866501,ok,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,0.931888,0.740821,0.979954,0.2462,0.123937,0.911740,0.462958,0.304837
7,BayesAdapter,cifar_10,sun397,SUN397,8,1,msp_uncertainty,10000,19850,0.133499,0.492954,0.866501,ok,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,0.872883,0.921722,0.724527,0.6127,0.186313,0.911740,0.462958,0.304837


Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/bayesrt_vs_bayesadapter_msp_uncertainty_rc_analysis/summaries/summary_ood_msp_mean_std.csv
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/bayesrt_vs_bayesadapter_msp_uncertainty_rc_analysis/summaries/summary_ood_msp_delta.csv
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/bayesrt_vs_bayesadapter_msp_uncertainty_rc_analysis/summaries/msp_uncertainty_distribution_figures.csv
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/bayesrt_vs_bayesadapter_msp_uncertainty_rc_analysis/summaries/msp_correct_wrong_ood_figures.csv
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/bayesrt_vs_bayesadapter_msp_uncertainty_rc_analysis/summaries/feature_embedding_R_C_adapter_figures.csv
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/bayesrt_vs_bayesadapter_msp_uncertainty_rc_analysis/summaries/similarity_matrix_R_C_adapter_figures.csv
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/bayesrt_vs_bayesadapter_msp_uncertai

## 9. mean/std 与 delta 汇总

`delta = BayesRTMMRL - BayesAdapter`。

注意指标方向：

- `AUROC`, `AUPR_OUT`, `AUPR_IN`, `ErrorAUROC`, `ErrorAUPR`：越高越好。
- `FPR95`, `DetectionError`, `ErrorFPR95`, `ECE`, `NLL`, `Brier`, `AURC`, `EAURC`：越低越好。

In [10]:

# =========================
# 9. 汇总表
# =========================
HIGHER_BETTER_METRICS = ["AUROC", "AUPR_OUT", "AUPR_IN", "ErrorAUROC", "ErrorAUPR", "Accuracy"]
LOWER_BETTER_METRICS = ["FPR95", "DetectionError", "ErrorFPR95", "ECE", "MCE", "NLL", "Brier", "AURC", "EAURC"]
OOD_METRIC_COLS = ["AUROC", "AUPR_OUT", "AUPR_IN", "FPR95", "DetectionError", "ErrorAUROC", "ErrorAUPR", "ErrorFPR95"]


def metric_direction(metric):
    if metric in HIGHER_BETTER_METRICS:
        return "higher_better"
    if metric in LOWER_BETTER_METRICS:
        return "lower_better"
    return "unknown"


def better_method_from_delta(delta, metric):
    if pd.isna(delta):
        return "unknown"
    direction = metric_direction(metric)
    if direction == "higher_better":
        return "BayesRTMMRL" if delta > 0 else ("BayesAdapter" if delta < 0 else "tie")
    if direction == "lower_better":
        return "BayesRTMMRL" if delta < 0 else ("BayesAdapter" if delta > 0 else "tie")
    return "unknown"


def mean_std_summary(df: pd.DataFrame, metric_cols):
    ok = df[df.get("status", "ok").eq("ok")].copy()
    if ok.empty:
        return pd.DataFrame()

    group_cols = ["method", "id_dataset", "ood_dataset", "shot", "score_name"]
    rows = []
    for keys, g in ok.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, keys))
        row["num_seeds"] = int(g["seed"].nunique())
        row["seeds"] = " ".join(str(int(x)) for x in sorted(g["seed"].dropna().unique()))
        for m in metric_cols:
            if m in g.columns:
                vals = pd.to_numeric(g[m], errors="coerce").dropna()
                row[f"{m}_mean"] = float(vals.mean()) if len(vals) else np.nan
                row[f"{m}_std"] = float(vals.std(ddof=0)) if len(vals) > 1 else (0.0 if len(vals) == 1 else np.nan)
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["id_dataset", "ood_dataset", "shot", "method"])


def delta_summary(df_mean: pd.DataFrame, metric_cols):
    if df_mean.empty:
        return pd.DataFrame()

    id_cols = ["id_dataset", "ood_dataset", "shot", "score_name"]
    rows = []
    for keys, g in df_mean.groupby(id_cols, dropna=False):
        base = dict(zip(id_cols, keys))
        rt = g[g["method"].eq("BayesRTMMRL")]
        ba = g[g["method"].eq("BayesAdapter")]
        if rt.empty or ba.empty:
            row = {**base, "status": "missing_pair"}
            rows.append(row)
            continue
        for m in metric_cols:
            rt_val = float(rt.iloc[0].get(f"{m}_mean", np.nan))
            ba_val = float(ba.iloc[0].get(f"{m}_mean", np.nan))
            delta = rt_val - ba_val
            rows.append({
                **base,
                "metric": m,
                "metric_direction": metric_direction(m),
                "BayesRTMMRL": rt_val,
                "BayesAdapter": ba_val,
                "delta_BayesRTMMRL_minus_BayesAdapter": delta,
                "better_method": better_method_from_delta(delta, m),
                "status": "ok",
            })
    return pd.DataFrame(rows).sort_values(["id_dataset", "ood_dataset", "shot", "metric"])

ood_mean_std = mean_std_summary(ood_raw, OOD_METRIC_COLS)
ood_mean_std_csv = SUMMARY_ROOT / "summary_ood_msp_mean_std.csv"
ood_mean_std.to_csv(ood_mean_std_csv, index=False)

ood_delta = delta_summary(ood_mean_std, OOD_METRIC_COLS)
ood_delta_csv = SUMMARY_ROOT / "summary_ood_msp_delta.csv"
ood_delta.to_csv(ood_delta_csv, index=False)

print("Saved:", ood_mean_std_csv)
print("Saved:", ood_delta_csv)
display(ood_mean_std.head(30))
display(ood_delta.head(30))

,method,id_dataset,ood_dataset,shot,score_name,num_seeds,seeds,AUROC_mean,AUROC_std,AUPR_OUT_mean,AUPR_OUT_std,AUPR_IN_mean,AUPR_IN_std,FPR95_mean,FPR95_std,DetectionError_mean,DetectionError_std,ErrorAUROC_mean,ErrorAUROC_std,ErrorAUPR_mean,ErrorAUPR_std,ErrorFPR95_mean,ErrorFPR95_std
0,BayesAdapter,cifar_10,dtd,8,msp_uncertainty,1,1,0.947682,0.0,0.719313,0.0,0.990322,0.0,0.2038,0.0,0.118150,0.0,0.911740,0.0,0.462958,0.0,0.304837,0.0
4,BayesRTMMRL,cifar_10,dtd,8,msp_uncertainty,1,1,0.971709,0.0,0.864790,0.0,0.994594,0.0,0.1156,0.0,0.079592,0.0,0.907107,0.0,0.380174,0.0,0.313013,0.0
1,BayesAdapter,cifar_10,oxford_flowers,8,msp_uncertainty,1,1,0.931888,0.0,0.740821,0.0,0.979954,0.0,0.2462,0.0,0.123937,0.0,0.911740,0.0,0.462958,0.0,0.304837,0.0
5,BayesRTMMRL,cifar_10,oxford_flowers,8,msp_uncertainty,1,1,0.978593,0.0,0.909995,0.0,0.994888,0.0,0.0963,0.0,0.072252,0.0,0.907107,0.0,0.380174,0.0,0.313013,0.0
2,BayesAdapter,cifar_10,sun397,8,msp_uncertainty,1,1,0.872883,0.0,0.921722,0.0,0.724527,0.0,0.6127,0.0,0.186313,0.0,0.911740,0.0,0.462958,0.0,0.304837,0.0
6,BayesRTMMRL,cifar_10,sun397,8,msp_uncertainty,1,1,0.936290,0.0,0.964003,0.0,0.877175,0.0,0.2608,0.0,0.127326,0.0,0.907107,0.0,0.380174,0.0,0.313013,0.0
3,BayesAdapter,cifar_10,tinyimagenet,8,msp_uncertainty,1,1,0.860875,0.0,0.861882,0.0,0.812059,0.0,0.7105,0.0,0.197400,0.0,0.911740,0.0,0.462958,0.0,0.304837,0.0
7,BayesRTMMRL,cifar_10,tinyimagenet,8,msp_uncertainty,1,1,0.905695,0.0,0.911133,0.0,0.886796,0.0,0.4643,0.0,0.161950,0.0,0.907107,0.0,0.380174,0.0,0.313013,0.0


,id_dataset,ood_dataset,shot,score_name,metric,metric_direction,BayesRTMMRL,BayesAdapter,delta_BayesRTMMRL_minus_BayesAdapter,better_method,status
2,cifar_10,dtd,8,msp_uncertainty,AUPR_IN,higher_better,0.994594,0.990322,0.004272,BayesRTMMRL,ok
1,cifar_10,dtd,8,msp_uncertainty,AUPR_OUT,higher_better,0.864790,0.719313,0.145476,BayesRTMMRL,ok
0,cifar_10,dtd,8,msp_uncertainty,AUROC,higher_better,0.971709,0.947682,0.024027,BayesRTMMRL,ok
4,cifar_10,dtd,8,msp_uncertainty,DetectionError,lower_better,0.079592,0.118150,-0.038558,BayesRTMMRL,ok
6,cifar_10,dtd,8,msp_uncertainty,ErrorAUPR,higher_better,0.380174,0.462958,-0.082785,BayesAdapter,ok
5,cifar_10,dtd,8,msp_uncertainty,ErrorAUROC,higher_better,0.907107,0.911740,-0.004633,BayesAdapter,ok
7,cifar_10,dtd,8,msp_uncertainty,ErrorFPR95,lower_better,0.313013,0.304837,0.008176,BayesAdapter,ok
3,cifar_10,dtd,8,msp_uncertainty,FPR95,lower_better,0.115600,0.203800,-0.088200,BayesRTMMRL,ok
10,cifar_10,oxford_flowers,8,msp_uncertainty,AUPR_IN,higher_better,0.994888,0.979954,0.014934,BayesRTMMRL,ok
9,cifar_10,oxford_flowers,8,msp_uncertainty,AUPR_OUT,higher_better,0.909995,0.740821,0.169174,BayesRTMMRL,ok


## 10. 绘图工具：KDE / ECDF 曲线

分布图只能用曲线，不画柱状图。

In [11]:

# =========================
# 10. 绘图工具
# =========================
import matplotlib.pyplot as plt


def to_numpy_1d(x):
    if x is None:
        return np.array([], dtype=float)
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy().reshape(-1)
    return np.asarray(x).reshape(-1)


def density_curve(values, x_grid=None, value_range=(0.0, 1.0), num=256):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if x_grid is None:
        x_grid = np.linspace(value_range[0], value_range[1], num)
    if len(vals) < 2:
        return x_grid, np.zeros_like(x_grid)
    if SCIPY_AVAILABLE:
        try:
            kde = gaussian_kde(vals)
            y = kde(x_grid)
            return x_grid, y
        except Exception:
            pass
    # Fallback: line-smoothed histogram density, still plotted as a curve.
    counts, edges = np.histogram(vals, bins=min(80, max(10, int(np.sqrt(len(vals))))), range=value_range, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    y = np.interp(x_grid, centers, counts, left=0.0, right=0.0)
    kernel = np.ones(5) / 5
    y = np.convolve(y, kernel, mode="same")
    return x_grid, y


def ecdf_curve(values):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    vals = np.sort(vals)
    if len(vals) == 0:
        return vals, vals
    y = np.arange(1, len(vals) + 1) / len(vals)
    return vals, y


def plot_distribution_line(ax, values, label, mode="KDE", value_range=(0.0, 1.0), linestyle="-", linewidth=1.8):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return
    if str(mode).upper() == "ECDF":
        x, y = ecdf_curve(vals)
        ax.plot(x, y, label=label, linestyle=linestyle, linewidth=linewidth)
        ax.set_ylabel("ECDF")
    else:
        x, y = density_curve(vals, value_range=value_range)
        ax.plot(x, y, label=label, linestyle=linestyle, linewidth=linewidth)
        ax.set_ylabel("Density")


def load_case_payloads(method, id_dataset, shot, seed, ood_dataset):
    cache_dir = build_case_cache_dir(method, id_dataset, shot, seed)
    id_path = cache_dir / "id_test_outputs.pt"
    ood_path = cache_dir / f"ood_{ood_dataset}_outputs.pt"
    if not id_path.exists() or not ood_path.exists():
        return None, None
    return load_tensor_payload(id_path), load_tensor_payload(ood_path)


def get_unc(payload):
    return to_numpy_1d(payload.get("msp_uncertainty", None))


def get_labels(payload):
    labels = payload.get("labels", None)
    return None if labels is None else to_numpy_1d(labels).astype(int)


def get_correct(payload):
    corr = payload.get("correct", None)
    return None if corr is None else to_numpy_1d(corr).astype(bool)

## 11. 主图：MSP uncertainty ID/OOD 分布曲线

每张 figure 固定：`ID dataset × OOD dataset × shot`。

每个 seed 是一个 subplot；每个 subplot 中同时画：

- `BayesRTMMRL - ID`
- `BayesRTMMRL - OOD`
- `BayesAdapter - ID`
- `BayesAdapter - OOD`

In [12]:

# =========================
# 11. MSP uncertainty ID/OOD seed panels
# =========================
def plot_msp_uncertainty_seed_panels(id_dataset, ood_dataset, shot, mode="KDE"):
    fig, axes = plt.subplots(1, len(SEEDS), figsize=(5.4 * len(SEEDS), 4.2), sharex=True, sharey=True)
    if len(SEEDS) == 1:
        axes = [axes]

    any_plotted = False
    for ax, seed in zip(axes, SEEDS):
        for method, linestyle in [("BayesRTMMRL", "-"), ("BayesAdapter", "--")]:
            id_payload, ood_payload = load_case_payloads(method, id_dataset, shot, seed, ood_dataset)
            if id_payload is None or ood_payload is None:
                continue
            id_unc = get_unc(id_payload)
            ood_unc = get_unc(ood_payload)
            plot_distribution_line(ax, id_unc, f"{method} ID", mode=mode, linestyle=linestyle)
            plot_distribution_line(ax, ood_unc, f"{method} OOD", mode=mode, linestyle=linestyle)
            any_plotted = True

        ax.set_title(f"seed={seed}")
        ax.set_xlabel("MSP uncertainty = 1 - max softmax probability")
        ax.set_xlim(0.0, 1.0)
        ax.grid(alpha=0.25)

    axes[0].legend(fontsize=8)
    fig.suptitle(f"MSP uncertainty distribution: {id_dataset} vs {ood_dataset}, shot={shot}")
    fig.tight_layout()
    if not any_plotted:
        plt.close(fig)
        return None

    out_dir = FIG_DIRS["msp_kde"] if str(mode).upper() == "KDE" else FIG_DIRS["msp_ecdf"]
    out_path = out_dir / f"msp_uncertainty_{str(mode).lower()}_{id_dataset}_{ood_dataset}_shot{shot}_seed_panels.png"
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    return str(out_path)

msp_fig_rows = []
for id_dataset in ID_DATASETS:
    for ood_dataset in OOD_DATASETS:
        for shot in SHOTS:
            p = plot_msp_uncertainty_seed_panels(id_dataset, ood_dataset, shot, mode=DISTRIBUTION_PLOT_TYPE)
            if p:
                msp_fig_rows.append({"id_dataset": id_dataset, "ood_dataset": ood_dataset, "shot": shot, "mode": DISTRIBUTION_PLOT_TYPE, "figure": p})

msp_fig_df = pd.DataFrame(msp_fig_rows)
msp_fig_csv = SUMMARY_ROOT / "msp_uncertainty_distribution_figures.csv"
msp_fig_df.to_csv(msp_fig_csv, index=False)
print("Saved:", msp_fig_csv)
display(msp_fig_df.head(20))

,id_dataset,ood_dataset,shot,mode,figure
0,cifar_10,dtd,8,KDE,/root/autodl-tmp/MMRL/output_refactor/analysis...
1,cifar_10,tinyimagenet,8,KDE,/root/autodl-tmp/MMRL/output_refactor/analysis...
2,cifar_10,oxford_flowers,8,KDE,/root/autodl-tmp/MMRL/output_refactor/analysis...
3,cifar_10,sun397,8,KDE,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 12. ID correct / ID wrong / OOD 的 MSP uncertainty 曲线

每张 figure 固定：`ID dataset × OOD dataset × shot`。

布局：

```text
rows = seeds
columns = methods
```

每个 panel 中画三条曲线：`ID correct`、`ID wrong`、`OOD`。

In [13]:

# =========================
# 12. Correct / Wrong / OOD uncertainty panels
# =========================
def plot_correct_wrong_ood_panels(id_dataset, ood_dataset, shot, mode="KDE"):
    methods = ["BayesRTMMRL", "BayesAdapter"]
    fig, axes = plt.subplots(len(SEEDS), len(methods), figsize=(5.4 * len(methods), 3.8 * len(SEEDS)), sharex=True, sharey=True)
    if len(SEEDS) == 1:
        axes = np.asarray([axes])
    if len(methods) == 1:
        axes = axes[:, None]

    any_plotted = False
    for r, seed in enumerate(SEEDS):
        for c, method in enumerate(methods):
            ax = axes[r, c]
            id_payload, ood_payload = load_case_payloads(method, id_dataset, shot, seed, ood_dataset)
            if id_payload is not None and ood_payload is not None:
                id_unc = get_unc(id_payload)
                ood_unc = get_unc(ood_payload)
                correct = get_correct(id_payload)
                if correct is not None and len(correct) == len(id_unc):
                    plot_distribution_line(ax, id_unc[correct], "ID correct", mode=mode, linestyle="-")
                    plot_distribution_line(ax, id_unc[~correct], "ID wrong", mode=mode, linestyle="--")
                else:
                    plot_distribution_line(ax, id_unc, "ID", mode=mode, linestyle="-")
                plot_distribution_line(ax, ood_unc, "OOD", mode=mode, linestyle=":")
                any_plotted = True
            ax.set_title(f"{method}, seed={seed}")
            ax.set_xlabel("MSP uncertainty")
            ax.set_xlim(0.0, 1.0)
            ax.grid(alpha=0.25)
            if r == 0 and c == 0:
                ax.legend(fontsize=8)

    fig.suptitle(f"ID correct / ID wrong / OOD MSP uncertainty: {id_dataset} vs {ood_dataset}, shot={shot}")
    fig.tight_layout()
    if not any_plotted:
        plt.close(fig)
        return None
    out_path = FIG_DIRS["msp_cwo"] / f"msp_correct_wrong_ood_{id_dataset}_{ood_dataset}_shot{shot}_seed_method_panels.png"
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    return str(out_path)

cwo_fig_rows = []
for id_dataset in ID_DATASETS:
    for ood_dataset in OOD_DATASETS:
        for shot in SHOTS:
            p = plot_correct_wrong_ood_panels(id_dataset, ood_dataset, shot, mode=DISTRIBUTION_PLOT_TYPE)
            if p:
                cwo_fig_rows.append({"id_dataset": id_dataset, "ood_dataset": ood_dataset, "shot": shot, "mode": DISTRIBUTION_PLOT_TYPE, "figure": p})

cwo_fig_df = pd.DataFrame(cwo_fig_rows)
cwo_fig_csv = SUMMARY_ROOT / "msp_correct_wrong_ood_figures.csv"
cwo_fig_df.to_csv(cwo_fig_csv, index=False)
print("Saved:", cwo_fig_csv)
display(cwo_fig_df.head(20))

,id_dataset,ood_dataset,shot,mode,figure
0,cifar_10,dtd,8,KDE,/root/autodl-tmp/MMRL/output_refactor/analysis...
1,cifar_10,tinyimagenet,8,KDE,/root/autodl-tmp/MMRL/output_refactor/analysis...
2,cifar_10,oxford_flowers,8,KDE,/root/autodl-tmp/MMRL/output_refactor/analysis...
3,cifar_10,sun397,8,KDE,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 13. 特征采样与 embedding 工具

BayesRTMMRL 需要分别画：

- `R branch`
- `C branch`

BayesAdapter 画：

- `adapter feature`

每个 seed 单独导出一张图，列布局为：

```text
BayesRTMMRL-R | BayesRTMMRL-C | BayesAdapter
```

In [14]:

# =========================
# 13. 特征采样与 embedding 工具
# =========================
def get_feature_np(payload, branch):
    feats = payload.get("features", {})
    if not isinstance(feats, dict) or branch not in feats:
        return None
    x = feats[branch]
    if isinstance(x, torch.Tensor):
        return x.detach().float().cpu().numpy()
    return np.asarray(x, dtype=np.float32)


def sample_id_indices_by_class(labels, max_per_class=80, random_state=2026):
    rng = np.random.default_rng(random_state)
    labels = np.asarray(labels).astype(int)
    indices = []
    for y in sorted(np.unique(labels)):
        idx = np.where(labels == y)[0]
        if len(idx) > max_per_class:
            idx = rng.choice(idx, size=max_per_class, replace=False)
        indices.extend(idx.tolist())
    return np.asarray(sorted(indices), dtype=int)


def sample_ood_indices(n, max_total=800, random_state=2026):
    rng = np.random.default_rng(random_state)
    idx = np.arange(n)
    if n > max_total:
        idx = rng.choice(idx, size=max_total, replace=False)
    return np.asarray(sorted(idx), dtype=int)


def compute_2d_embedding(x, method="umap", random_state=2026):
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x)
    n = len(x)
    if n == 0:
        return np.zeros((0, 2), dtype=np.float32)

    # PCA 预降维，提升 t-SNE/UMAP 稳定性。
    if SKLEARN_EMBED_AVAILABLE and x.shape[1] > 50:
        x_reduced = PCA(n_components=50, random_state=random_state).fit_transform(x)
    else:
        x_reduced = x

    if method.lower() == "umap" and UMAP_AVAILABLE:
        reducer = umap.UMAP(n_components=2, random_state=random_state, n_neighbors=15, min_dist=0.1, metric="cosine")
        return reducer.fit_transform(x_reduced)

    if method.lower() == "tsne" and SKLEARN_EMBED_AVAILABLE:
        perplexity = min(30, max(5, (n - 1) // 3))
        return TSNE(n_components=2, random_state=random_state, init="pca", learning_rate="auto", perplexity=perplexity).fit_transform(x_reduced)

    if SKLEARN_EMBED_AVAILABLE:
        return PCA(n_components=2, random_state=random_state).fit_transform(x_reduced)

    # Last-resort fallback: first two dimensions.
    if x.shape[1] >= 2:
        return x[:, :2]
    return np.c_[x[:, 0], np.zeros(n)]


def build_embedding_data(id_payload, ood_payload, branch, random_state=2026):
    x_id = get_feature_np(id_payload, branch)
    x_ood = get_feature_np(ood_payload, branch)
    if x_id is None or x_ood is None:
        return None
    labels = get_labels(id_payload)
    if labels is None:
        labels = np.zeros(len(x_id), dtype=int)

    id_idx = sample_id_indices_by_class(labels, MAX_ID_PER_CLASS_FOR_FEATURE, random_state=random_state)
    ood_idx = sample_ood_indices(len(x_ood), MAX_OOD_FOR_FEATURE, random_state=random_state)

    x = np.vstack([x_id[id_idx], x_ood[ood_idx]])
    domain = np.array(["ID"] * len(id_idx) + ["OOD"] * len(ood_idx))
    y = np.r_[labels[id_idx], np.array([-1] * len(ood_idx))]
    emb = compute_2d_embedding(l2_normalize_np(x), method="umap" if UMAP_AVAILABLE else "tsne", random_state=random_state)
    return {"emb": emb, "domain": domain, "labels": y}

## 14. 特征 embedding 图：R / C / Adapter

每个 seed 单独生成一张图。

In [15]:

# =========================
# 14. Feature embedding plots
# =========================
def plot_feature_embedding_rc_adapter(id_dataset, ood_dataset, shot, seed):
    panels = [
        ("BayesRTMMRL", "R", "BayesRTMMRL - R branch"),
        ("BayesRTMMRL", "C", "BayesRTMMRL - C branch"),
        ("BayesAdapter", "adapter", "BayesAdapter"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.0), sharex=False, sharey=False)
    any_plotted = False

    for ax, (method, branch, title) in zip(axes, panels):
        id_payload, ood_payload = load_case_payloads(method, id_dataset, shot, seed, ood_dataset)
        if id_payload is None or ood_payload is None:
            ax.set_title(title + "\nmissing payload")
            ax.axis("off")
            continue
        data = build_embedding_data(id_payload, ood_payload, branch, random_state=RANDOM_STATE + int(seed))
        if data is None:
            ax.set_title(title + "\nmissing feature")
            ax.axis("off")
            continue

        emb = data["emb"]
        domain = data["domain"]
        labels = data["labels"]
        id_mask = domain == "ID"
        ood_mask = domain == "OOD"

        # ID by class. 不指定具体颜色，使用 matplotlib 默认循环。
        for y in sorted(np.unique(labels[id_mask])):
            m = id_mask & (labels == y)
            ax.scatter(emb[m, 0], emb[m, 1], s=8, alpha=0.65, label=f"ID {y}")
        ax.scatter(emb[ood_mask, 0], emb[ood_mask, 1], s=10, alpha=0.75, marker="x", label="OOD")
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
        any_plotted = True

    # 只放一个简化图例，避免过密。
    handles, labels_ = axes[-1].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels_, loc="center right", fontsize=8, frameon=False)
    fig.suptitle(f"Feature embedding: {id_dataset} vs {ood_dataset}, shot={shot}, seed={seed}")
    fig.tight_layout(rect=[0, 0, 0.92, 1])

    if not any_plotted:
        plt.close(fig)
        return None
    out_path = FIG_DIRS["feature_embedding"] / f"embedding_{id_dataset}_{ood_dataset}_shot{shot}_seed{seed}_R_C_adapter.png"
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    return str(out_path)

embedding_fig_rows = []
for id_dataset in ID_DATASETS:
    for ood_dataset in OOD_DATASETS:
        for shot in SHOTS:
            for seed in SEEDS:
                p = plot_feature_embedding_rc_adapter(id_dataset, ood_dataset, shot, seed)
                if p:
                    embedding_fig_rows.append({"id_dataset": id_dataset, "ood_dataset": ood_dataset, "shot": shot, "seed": seed, "figure": p})

embedding_fig_df = pd.DataFrame(embedding_fig_rows)
embedding_fig_csv = SUMMARY_ROOT / "feature_embedding_R_C_adapter_figures.csv"
embedding_fig_df.to_csv(embedding_fig_csv, index=False)
print("Saved:", embedding_fig_csv)
display(embedding_fig_df.head(20))

,id_dataset,ood_dataset,shot,seed,figure
0,cifar_10,dtd,8,1,/root/autodl-tmp/MMRL/output_refactor/analysis...
1,cifar_10,tinyimagenet,8,1,/root/autodl-tmp/MMRL/output_refactor/analysis...
2,cifar_10,oxford_flowers,8,1,/root/autodl-tmp/MMRL/output_refactor/analysis...
3,cifar_10,sun397,8,1,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 15. 特征相似度矩阵与统计

每个 seed 单独一张 figure。

布局：

```text
                 ID-ID        OOD-OOD       ID-OOD
BayesRTMMRL-R    heatmap      heatmap       heatmap
BayesRTMMRL-C    heatmap      heatmap       heatmap
BayesAdapter     heatmap      heatmap       heatmap
```

所有 heatmap 使用相同 color scale。

In [16]:

# =========================
# 15. Similarity matrix utilities
# =========================
def sample_for_similarity(id_payload, ood_payload, branch, random_state=2026):
    x_id = get_feature_np(id_payload, branch)
    x_ood = get_feature_np(ood_payload, branch)
    if x_id is None or x_ood is None:
        return None
    labels = get_labels(id_payload)
    if labels is None:
        labels = np.zeros(len(x_id), dtype=int)

    id_idx = sample_id_indices_by_class(labels, SIM_MATRIX_ID_PER_CLASS, random_state=random_state)
    ood_idx = sample_ood_indices(len(x_ood), SIM_MATRIX_OOD_TOTAL, random_state=random_state)

    # ID 按类别排序。
    order = np.argsort(labels[id_idx])
    id_idx = id_idx[order]

    zid = l2_normalize_np(x_id[id_idx])
    zood = l2_normalize_np(x_ood[ood_idx])
    yid = labels[id_idx]

    return {
        "zid": zid,
        "zood": zood,
        "yid": yid,
        "id_idx": id_idx,
        "ood_idx": ood_idx,
    }


def cosine_matrix(a, b):
    return np.asarray(a, dtype=np.float64) @ np.asarray(b, dtype=np.float64).T


def compute_similarity_pack(id_payload, ood_payload, branch, random_state=2026):
    data = sample_for_similarity(id_payload, ood_payload, branch, random_state=random_state)
    if data is None:
        return None
    zid, zood, yid = data["zid"], data["zood"], data["yid"]
    s_id_id = cosine_matrix(zid, zid)
    s_ood_ood = cosine_matrix(zood, zood)
    s_id_ood = cosine_matrix(zid, zood)

    same = yid[:, None] == yid[None, :]
    not_diag = ~np.eye(len(yid), dtype=bool)
    same_no_diag = same & not_diag
    diff = (~same) & not_diag

    stats = {
        "mean_sim_id_id": float(s_id_id[not_diag].mean()) if not_diag.any() else np.nan,
        "mean_sim_id_same_class": float(s_id_id[same_no_diag].mean()) if same_no_diag.any() else np.nan,
        "mean_sim_id_diff_class": float(s_id_id[diff].mean()) if diff.any() else np.nan,
        "mean_sim_ood_ood": float(s_ood_ood[~np.eye(len(zood), dtype=bool)].mean()) if len(zood) > 1 else np.nan,
        "mean_sim_id_ood": float(s_id_ood.mean()) if s_id_ood.size else np.nan,
    }
    stats["gap_same_vs_diff"] = stats["mean_sim_id_same_class"] - stats["mean_sim_id_diff_class"]
    stats["gap_idid_vs_idood"] = stats["mean_sim_id_id"] - stats["mean_sim_id_ood"]

    return {
        "S_ID_ID": s_id_id,
        "S_OOD_OOD": s_ood_ood,
        "S_ID_OOD": s_id_ood,
        "yid": yid,
        "stats": stats,
    }


def plot_similarity_heatmaps(id_dataset, ood_dataset, shot, seed):
    panels = [
        ("BayesRTMMRL", "R", "BayesRTMMRL-R"),
        ("BayesRTMMRL", "C", "BayesRTMMRL-C"),
        ("BayesAdapter", "adapter", "BayesAdapter"),
    ]
    cols = [("S_ID_ID", "ID-ID"), ("S_OOD_OOD", "OOD-OOD"), ("S_ID_OOD", "ID-OOD")]
    fig, axes = plt.subplots(3, 3, figsize=(13.2, 12.0))
    any_plotted = False
    stat_rows = []

    for r, (method, branch, row_title) in enumerate(panels):
        id_payload, ood_payload = load_case_payloads(method, id_dataset, shot, seed, ood_dataset)
        pack = None
        if id_payload is not None and ood_payload is not None:
            pack = compute_similarity_pack(id_payload, ood_payload, branch, random_state=RANDOM_STATE + int(seed))
        if pack is not None:
            stat_rows.append({
                "method": method,
                "feature_branch": branch,
                "id_dataset": id_dataset,
                "ood_dataset": ood_dataset,
                "shot": int(shot),
                "seed": int(seed),
                **pack["stats"],
            })
        for c, (key, col_title) in enumerate(cols):
            ax = axes[r, c]
            if pack is None:
                ax.text(0.5, 0.5, "missing feature", ha="center", va="center")
                ax.set_axis_off()
                continue
            im = ax.imshow(pack[key], aspect="auto", vmin=-1.0, vmax=1.0)
            ax.set_title(f"{row_title}: {col_title}")
            ax.set_xticks([])
            ax.set_yticks([])
            any_plotted = True

    fig.suptitle(f"Cosine similarity matrices: {id_dataset} vs {ood_dataset}, shot={shot}, seed={seed}")
    fig.tight_layout(rect=[0, 0.03, 0.92, 1])
    cbar_ax = fig.add_axes([0.94, 0.15, 0.015, 0.7])
    fig.colorbar(im if any_plotted else axes[0, 0].imshow([[0]], vmin=-1, vmax=1), cax=cbar_ax)

    fig_path = None
    if any_plotted:
        fig_path = FIG_DIRS["similarity_matrix"] / f"sim_matrix_{id_dataset}_{ood_dataset}_shot{shot}_seed{seed}_R_C_adapter.png"
        fig.savefig(fig_path, dpi=220)
    plt.close(fig)
    return (str(fig_path) if fig_path else None), stat_rows

sim_fig_rows = []
sim_stat_rows = []
for id_dataset in ID_DATASETS:
    for ood_dataset in OOD_DATASETS:
        for shot in SHOTS:
            for seed in SEEDS:
                p, rows = plot_similarity_heatmaps(id_dataset, ood_dataset, shot, seed)
                if p:
                    sim_fig_rows.append({"id_dataset": id_dataset, "ood_dataset": ood_dataset, "shot": shot, "seed": seed, "figure": p})
                sim_stat_rows.extend(rows)

sim_fig_df = pd.DataFrame(sim_fig_rows)
sim_fig_csv = SUMMARY_ROOT / "similarity_matrix_R_C_adapter_figures.csv"
sim_fig_df.to_csv(sim_fig_csv, index=False)

similarity_raw = pd.DataFrame(sim_stat_rows)
sim_raw_csv = SUMMARY_ROOT / "summary_similarity_raw.csv"
similarity_raw.to_csv(sim_raw_csv, index=False)

print("Saved:", sim_fig_csv)
print("Saved:", sim_raw_csv)
display(similarity_raw.head(20))

,method,feature_branch,id_dataset,ood_dataset,shot,seed,mean_sim_id_id,mean_sim_id_same_class,mean_sim_id_diff_class,mean_sim_ood_ood,mean_sim_id_ood,gap_same_vs_diff,gap_idid_vs_idood
0,BayesRTMMRL,R,cifar_10,dtd,8,1,0.209694,0.887914,0.135844,0.713440,0.313993,0.752070,-0.104298
1,BayesRTMMRL,C,cifar_10,dtd,8,1,0.761053,0.854701,0.750856,0.662895,0.589523,0.103846,0.171530
2,BayesAdapter,adapter,cifar_10,dtd,8,1,0.787766,0.854721,0.780476,0.657157,0.603152,0.074246,0.184614
3,BayesRTMMRL,R,cifar_10,tinyimagenet,8,1,0.209694,0.887914,0.135844,0.544481,0.287903,0.752070,-0.078208
4,BayesRTMMRL,C,cifar_10,tinyimagenet,8,1,0.761053,0.854701,0.750856,0.678939,0.671040,0.103846,0.090013
5,BayesAdapter,adapter,cifar_10,tinyimagenet,8,1,0.787766,0.854721,0.780476,0.679434,0.685045,0.074246,0.102721
6,BayesRTMMRL,R,cifar_10,oxford_flowers,8,1,0.209694,0.887914,0.135844,0.859401,0.346316,0.752070,-0.136622
7,BayesRTMMRL,C,cifar_10,oxford_flowers,8,1,0.761053,0.854701,0.750856,0.794948,0.579834,0.103846,0.181219
8,BayesAdapter,adapter,cifar_10,oxford_flowers,8,1,0.787766,0.854721,0.780476,0.784066,0.577372,0.074246,0.210394
9,BayesRTMMRL,R,cifar_10,sun397,8,1,0.209694,0.887914,0.135844,0.726656,0.291233,0.752070,-0.081538


## 16. Similarity statistics mean/std/delta

`feature_branch` 取值包括：`R`、`C`、`adapter`。

这里的 delta 分两类：

1. R vs adapter
2. C vs adapter

用于分别回答：R/C 分支相对 BayesAdapter 在相似度结构上的差异。

**v3 修复**：`similarity_delta_vs_adapter()` 现在会在 BayesAdapter 或 R/C 分支缺失时返回带标准列的空表，而不是触发 `KeyError: id_dataset`；同时不再强制 BayesAdapter 的 `feature_branch` 必须严格等于 `adapter`。


In [17]:
# =========================
# 16. Similarity statistics summaries
# =========================
SIM_METRIC_COLS = [
    "mean_sim_id_id",
    "mean_sim_id_same_class",
    "mean_sim_id_diff_class",
    "mean_sim_ood_ood",
    "mean_sim_id_ood",
    "gap_same_vs_diff",
    "gap_idid_vs_idood",
]

SIM_MEAN_STD_COLUMNS = ["method", "feature_branch", "id_dataset", "ood_dataset", "shot", "num_seeds"]
for _m in SIM_METRIC_COLS:
    SIM_MEAN_STD_COLUMNS.extend([f"{_m}_mean", f"{_m}_std"])

SIM_DELTA_COLUMNS = [
    "id_dataset",
    "ood_dataset",
    "shot",
    "comparison",
    "metric",
    "BayesRTMMRL_branch",
    "BayesRTMMRL_value",
    "BayesAdapter_branch",
    "BayesAdapter_value",
    "delta",
]


def empty_similarity_mean_std_df():
    return pd.DataFrame(columns=SIM_MEAN_STD_COLUMNS)


def empty_similarity_delta_df():
    return pd.DataFrame(columns=SIM_DELTA_COLUMNS)


def normalize_branch_name(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def similarity_mean_std(df):
    """Aggregate similarity statistics over seeds.

    Always returns a DataFrame with the expected columns, even when no usable
    similarity rows exist. This prevents downstream KeyError when some feature
    branches are missing.
    """
    if df is None or df.empty:
        return empty_similarity_mean_std_df()

    required = {"method", "feature_branch", "id_dataset", "ood_dataset", "shot", "seed"}
    missing = sorted(required - set(df.columns))
    if missing:
        print(f"[WARN] similarity_raw is missing required columns: {missing}")
        return empty_similarity_mean_std_df()

    rows = []
    group_cols = ["method", "feature_branch", "id_dataset", "ood_dataset", "shot"]
    for keys, g in df.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, keys))
        row["num_seeds"] = int(g["seed"].nunique())
        for m in SIM_METRIC_COLS:
            if m not in g.columns:
                vals = pd.Series(dtype=float)
            else:
                vals = pd.to_numeric(g[m], errors="coerce").dropna()
            row[f"{m}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{m}_std"] = float(vals.std(ddof=0)) if len(vals) > 1 else (0.0 if len(vals) == 1 else np.nan)
        rows.append(row)

    if not rows:
        return empty_similarity_mean_std_df()
    return pd.DataFrame(rows, columns=SIM_MEAN_STD_COLUMNS).sort_values(
        ["id_dataset", "ood_dataset", "shot", "method", "feature_branch"],
        kind="mergesort",
    )


def pick_adapter_row(g):
    """Select the BayesAdapter row for a group.

    Earlier versions required feature_branch == 'adapter'. In practice, feature
    extraction may save BayesAdapter under a project-specific branch name. We
    therefore prefer 'adapter' when present, but fall back to the first
    BayesAdapter branch so delta generation does not silently fail.
    """
    adapter_rows = g[g["method"] == "BayesAdapter"].copy()
    if adapter_rows.empty:
        return None

    branch_norm = adapter_rows["feature_branch"].map(lambda x: normalize_branch_name(x).lower())
    preferred = adapter_rows[branch_norm == "adapter"]
    if not preferred.empty:
        return preferred.iloc[0]
    return adapter_rows.iloc[0]


def similarity_delta_vs_adapter(df_mean):
    """Compute BayesRTMMRL-R/C minus BayesAdapter deltas.

    Robust behavior:
    - returns an empty, well-formed DataFrame if no deltas can be computed;
    - does not raise KeyError when BayesAdapter features are missing;
    - accepts BayesAdapter feature branches with names other than exactly
      'adapter', while still recording the actual branch used.
    """
    if df_mean is None or df_mean.empty:
        return empty_similarity_delta_df()

    required = {"method", "feature_branch", "id_dataset", "ood_dataset", "shot"}
    missing = sorted(required - set(df_mean.columns))
    if missing:
        print(f"[WARN] similarity_mean is missing required columns: {missing}")
        return empty_similarity_delta_df()

    rows = []
    id_cols = ["id_dataset", "ood_dataset", "shot"]
    for keys, g in df_mean.groupby(id_cols, dropna=False):
        base = dict(zip(id_cols, keys))
        adapter_row = pick_adapter_row(g)
        if adapter_row is None:
            continue

        # Robust R/C branch match: exact after stripping, case-insensitive.
        branch_norm = g["feature_branch"].map(lambda x: normalize_branch_name(x).upper())
        for branch in ["R", "C"]:
            rt = g[(g["method"] == "BayesRTMMRL") & (branch_norm == branch)]
            if rt.empty:
                continue
            rt_row = rt.iloc[0]
            for m in SIM_METRIC_COLS:
                rt_val = float(rt_row.get(f"{m}_mean", np.nan))
                ad_val = float(adapter_row.get(f"{m}_mean", np.nan))
                rows.append({
                    **base,
                    "comparison": f"BayesRTMMRL-{branch}_minus_BayesAdapter",
                    "metric": m,
                    "BayesRTMMRL_branch": branch,
                    "BayesRTMMRL_value": rt_val,
                    "BayesAdapter_branch": adapter_row.get("feature_branch", "adapter"),
                    "BayesAdapter_value": ad_val,
                    "delta": rt_val - ad_val,
                })

    if not rows:
        print(
            "[WARN] No similarity deltas were produced. This usually means "
            "BayesAdapter features were not extracted, or BayesRTMMRL R/C branch "
            "features were missing. Check feature_key_report.csv and "
            "summary_similarity_mean_std.csv."
        )
        return empty_similarity_delta_df()

    return pd.DataFrame(rows, columns=SIM_DELTA_COLUMNS).sort_values(
        ["id_dataset", "ood_dataset", "shot", "comparison", "metric"],
        kind="mergesort",
    )


similarity_mean = similarity_mean_std(similarity_raw)
sim_mean_csv = SUMMARY_ROOT / "summary_similarity_mean_std.csv"
similarity_mean.to_csv(sim_mean_csv, index=False)

similarity_delta = similarity_delta_vs_adapter(similarity_mean)
sim_delta_csv = SUMMARY_ROOT / "summary_similarity_delta_vs_adapter.csv"
similarity_delta.to_csv(sim_delta_csv, index=False)

print("Saved:", sim_mean_csv)
print("Saved:", sim_delta_csv)
print("similarity_mean shape:", similarity_mean.shape)
print("similarity_delta shape:", similarity_delta.shape)
display(similarity_mean.head(30))
display(similarity_delta.head(30))


,method,feature_branch,id_dataset,ood_dataset,shot,num_seeds,mean_sim_id_id_mean,mean_sim_id_id_std,mean_sim_id_same_class_mean,mean_sim_id_same_class_std,mean_sim_id_diff_class_mean,mean_sim_id_diff_class_std,mean_sim_ood_ood_mean,mean_sim_ood_ood_std,mean_sim_id_ood_mean,mean_sim_id_ood_std,gap_same_vs_diff_mean,gap_same_vs_diff_std,gap_idid_vs_idood_mean,gap_idid_vs_idood_std
0,BayesAdapter,adapter,cifar_10,dtd,8,1,0.787766,0.0,0.854721,0.0,0.780476,0.0,0.657157,0.0,0.603152,0.0,0.074246,0.0,0.184614,0.0
4,BayesRTMMRL,C,cifar_10,dtd,8,1,0.761053,0.0,0.854701,0.0,0.750856,0.0,0.662895,0.0,0.589523,0.0,0.103846,0.0,0.171530,0.0
8,BayesRTMMRL,R,cifar_10,dtd,8,1,0.209694,0.0,0.887914,0.0,0.135844,0.0,0.713440,0.0,0.313993,0.0,0.752070,0.0,-0.104298,0.0
1,BayesAdapter,adapter,cifar_10,oxford_flowers,8,1,0.787766,0.0,0.854721,0.0,0.780476,0.0,0.784066,0.0,0.577372,0.0,0.074246,0.0,0.210394,0.0
5,BayesRTMMRL,C,cifar_10,oxford_flowers,8,1,0.761053,0.0,0.854701,0.0,0.750856,0.0,0.794948,0.0,0.579834,0.0,0.103846,0.0,0.181219,0.0
9,BayesRTMMRL,R,cifar_10,oxford_flowers,8,1,0.209694,0.0,0.887914,0.0,0.135844,0.0,0.859401,0.0,0.346316,0.0,0.752070,0.0,-0.136622,0.0
2,BayesAdapter,adapter,cifar_10,sun397,8,1,0.787766,0.0,0.854721,0.0,0.780476,0.0,0.524252,0.0,0.523077,0.0,0.074246,0.0,0.264689,0.0
6,BayesRTMMRL,C,cifar_10,sun397,8,1,0.761053,0.0,0.854701,0.0,0.750856,0.0,0.526901,0.0,0.515743,0.0,0.103846,0.0,0.245310,0.0
10,BayesRTMMRL,R,cifar_10,sun397,8,1,0.209694,0.0,0.887914,0.0,0.135844,0.0,0.726656,0.0,0.291233,0.0,0.752070,0.0,-0.081538,0.0
3,BayesAdapter,adapter,cifar_10,tinyimagenet,8,1,0.787766,0.0,0.854721,0.0,0.780476,0.0,0.679434,0.0,0.685045,0.0,0.074246,0.0,0.102721,0.0


,id_dataset,ood_dataset,shot,comparison,metric,BayesRTMMRL_branch,BayesRTMMRL_value,BayesAdapter_branch,BayesAdapter_value,delta
13,cifar_10,dtd,8,BayesRTMMRL-C_minus_BayesAdapter,gap_idid_vs_idood,C,0.171530,adapter,0.184614,-0.013083
12,cifar_10,dtd,8,BayesRTMMRL-C_minus_BayesAdapter,gap_same_vs_diff,C,0.103846,adapter,0.074246,0.029600
9,cifar_10,dtd,8,BayesRTMMRL-C_minus_BayesAdapter,mean_sim_id_diff_class,C,0.750856,adapter,0.780476,-0.029620
7,cifar_10,dtd,8,BayesRTMMRL-C_minus_BayesAdapter,mean_sim_id_id,C,0.761053,adapter,0.787766,-0.026713
11,cifar_10,dtd,8,BayesRTMMRL-C_minus_BayesAdapter,mean_sim_id_ood,C,0.589523,adapter,0.603152,-0.013630
8,cifar_10,dtd,8,BayesRTMMRL-C_minus_BayesAdapter,mean_sim_id_same_class,C,0.854701,adapter,0.854721,-0.000020
10,cifar_10,dtd,8,BayesRTMMRL-C_minus_BayesAdapter,mean_sim_ood_ood,C,0.662895,adapter,0.657157,0.005738
6,cifar_10,dtd,8,BayesRTMMRL-R_minus_BayesAdapter,gap_idid_vs_idood,R,-0.104298,adapter,0.184614,-0.288912
5,cifar_10,dtd,8,BayesRTMMRL-R_minus_BayesAdapter,gap_same_vs_diff,R,0.752070,adapter,0.074246,0.677824
2,cifar_10,dtd,8,BayesRTMMRL-R_minus_BayesAdapter,mean_sim_id_diff_class,R,0.135844,adapter,0.780476,-0.644632


## 17. Similarity distribution 曲线

每张 figure 固定：`ID dataset × OOD dataset × shot`。

布局：

```text
rows = seeds
columns = BayesRTMMRL-R / BayesRTMMRL-C / BayesAdapter
```

每个 panel 画：

- ID same-class similarity
- ID different-class similarity
- ID-OOD similarity

In [18]:

# =========================
# 17. Similarity distribution curves
# =========================
def similarity_vectors_from_pack(pack):
    s = pack["S_ID_ID"]
    y = pack["yid"]
    same = y[:, None] == y[None, :]
    not_diag = ~np.eye(len(y), dtype=bool)
    same_no_diag = same & not_diag
    diff = (~same) & not_diag
    return {
        "ID same-class": s[same_no_diag],
        "ID different-class": s[diff],
        "ID-OOD": pack["S_ID_OOD"].reshape(-1),
    }


def plot_similarity_distribution_panels(id_dataset, ood_dataset, shot):
    panels = [
        ("BayesRTMMRL", "R", "RTMMRL-R"),
        ("BayesRTMMRL", "C", "RTMMRL-C"),
        ("BayesAdapter", "adapter", "BayesAdapter"),
    ]
    fig, axes = plt.subplots(len(SEEDS), 3, figsize=(15.0, 3.8 * len(SEEDS)), sharex=True, sharey=False)
    if len(SEEDS) == 1:
        axes = np.asarray([axes])
    any_plotted = False

    for r, seed in enumerate(SEEDS):
        for c, (method, branch, title) in enumerate(panels):
            ax = axes[r, c]
            id_payload, ood_payload = load_case_payloads(method, id_dataset, shot, seed, ood_dataset)
            pack = None
            if id_payload is not None and ood_payload is not None:
                pack = compute_similarity_pack(id_payload, ood_payload, branch, random_state=RANDOM_STATE + int(seed))
            if pack is not None:
                vecs = similarity_vectors_from_pack(pack)
                for label, vals in vecs.items():
                    plot_distribution_line(ax, vals, label, mode="KDE", value_range=(-1.0, 1.0))
                any_plotted = True
            ax.set_title(f"{title}, seed={seed}")
            ax.set_xlabel("Cosine similarity")
            ax.set_xlim(-1.0, 1.0)
            ax.grid(alpha=0.25)
            if r == 0 and c == 0:
                ax.legend(fontsize=8)

    fig.suptitle(f"Similarity distributions: {id_dataset} vs {ood_dataset}, shot={shot}")
    fig.tight_layout()
    if not any_plotted:
        plt.close(fig)
        return None
    out_path = FIG_DIRS["similarity_distribution"] / f"sim_distribution_{id_dataset}_{ood_dataset}_shot{shot}_seed_panels_R_C_adapter.png"
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    return str(out_path)

sim_dist_fig_rows = []
for id_dataset in ID_DATASETS:
    for ood_dataset in OOD_DATASETS:
        for shot in SHOTS:
            p = plot_similarity_distribution_panels(id_dataset, ood_dataset, shot)
            if p:
                sim_dist_fig_rows.append({"id_dataset": id_dataset, "ood_dataset": ood_dataset, "shot": shot, "figure": p})

sim_dist_fig_df = pd.DataFrame(sim_dist_fig_rows)
sim_dist_fig_csv = SUMMARY_ROOT / "similarity_distribution_R_C_adapter_figures.csv"
sim_dist_fig_df.to_csv(sim_dist_fig_csv, index=False)
print("Saved:", sim_dist_fig_csv)
display(sim_dist_fig_df.head(20))

,id_dataset,ood_dataset,shot,figure
0,cifar_10,dtd,8,/root/autodl-tmp/MMRL/output_refactor/analysis...
1,cifar_10,tinyimagenet,8,/root/autodl-tmp/MMRL/output_refactor/analysis...
2,cifar_10,oxford_flowers,8,/root/autodl-tmp/MMRL/output_refactor/analysis...
3,cifar_10,sun397,8,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 18. ID calibration：reliability diagram 与 ECE/NLL/Brier

Calibration 本身必须基于 confidence，而不是 uncertainty，因为 reliability diagram 比较的是 `confidence` 与 `accuracy`。

这里只对 ID test set 做 calibration。

In [19]:

# =========================
# 18. Calibration metrics and reliability diagrams
# =========================
def calibration_metrics_from_payload(payload, n_bins=15):
    labels = payload.get("labels", None)
    logits = payload.get("logits", None)
    conf = payload.get("msp_confidence", None)
    preds = payload.get("preds", None)
    if labels is None or logits is None or conf is None or preds is None:
        return None
    y = to_numpy_1d(labels).astype(int)
    c = to_numpy_1d(conf).astype(float)
    pred = to_numpy_1d(preds).astype(int)
    correct = (pred == y).astype(float)
    probs = probs_from_logits_tensor(logits).detach().cpu().numpy()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    mce = 0.0
    bin_rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (c >= lo) & (c < hi if i < n_bins - 1 else c <= hi)
        if not mask.any():
            bin_rows.append({"bin": i, "lo": lo, "hi": hi, "count": 0, "confidence": np.nan, "accuracy": np.nan})
            continue
        acc = float(correct[mask].mean())
        avg_conf = float(c[mask].mean())
        gap = abs(acc - avg_conf)
        ece += float(mask.mean()) * gap
        mce = max(mce, gap)
        bin_rows.append({"bin": i, "lo": lo, "hi": hi, "count": int(mask.sum()), "confidence": avg_conf, "accuracy": acc})

    try:
        nll = float(log_loss(y, probs, labels=list(range(probs.shape[1]))))
    except Exception:
        nll = np.nan
    try:
        # Multi-class Brier: mean sum_c (p_c - onehot_c)^2
        onehot = np.eye(probs.shape[1])[y]
        brier = float(np.mean(np.sum((probs - onehot) ** 2, axis=1)))
    except Exception:
        brier = np.nan

    return {
        "ECE": float(ece),
        "MCE": float(mce),
        "NLL": nll,
        "Brier": brier,
        "Accuracy": float(correct.mean()),
        "AvgConfidence": float(c.mean()),
        "bin_rows": bin_rows,
    }

calib_rows = []
calib_bin_rows = []
for id_dataset in ID_DATASETS:
    for shot in SHOTS:
        for seed in SEEDS:
            for method in ["BayesRTMMRL", "BayesAdapter"]:
                cache_dir = build_case_cache_dir(method, id_dataset, shot, seed)
                id_path = cache_dir / "id_test_outputs.pt"
                if not id_path.exists():
                    continue
                payload = load_tensor_payload(id_path)
                m = calibration_metrics_from_payload(payload, n_bins=15)
                if m is None:
                    continue
                row = {"method": method, "id_dataset": id_dataset, "shot": int(shot), "seed": int(seed)}
                for k in ["ECE", "MCE", "NLL", "Brier", "Accuracy", "AvgConfidence"]:
                    row[k] = m[k]
                calib_rows.append(row)
                for br in m["bin_rows"]:
                    calib_bin_rows.append({**row, **br})

calibration_raw = pd.DataFrame(calib_rows)
calib_raw_csv = SUMMARY_ROOT / "summary_calibration_raw.csv"
calibration_raw.to_csv(calib_raw_csv, index=False)

calibration_bins = pd.DataFrame(calib_bin_rows)
calib_bins_csv = SUMMARY_ROOT / "summary_calibration_bins.csv"
calibration_bins.to_csv(calib_bins_csv, index=False)

calibration_mean = mean_std_summary(
    calibration_raw.assign(ood_dataset="", score_name="msp_confidence", status="ok"),
    ["ECE", "MCE", "NLL", "Brier", "Accuracy", "AvgConfidence"]
) if not calibration_raw.empty else pd.DataFrame()
calib_mean_csv = SUMMARY_ROOT / "summary_calibration_mean_std.csv"
calibration_mean.to_csv(calib_mean_csv, index=False)

print("Saved:", calib_raw_csv)
print("Saved:", calib_bins_csv)
print("Saved:", calib_mean_csv)
display(calibration_raw.head(20))

,method,id_dataset,shot,seed,ECE,MCE,NLL,Brier,Accuracy,AvgConfidence
0,BayesRTMMRL,cifar_10,8,1,0.014197,0.199918,0.201590,0.097548,0.9383,0.947240
1,BayesAdapter,cifar_10,8,1,0.057624,0.245961,0.271397,0.121572,0.9241,0.866501


In [20]:

# =========================
# 18b. Reliability diagram seed panels
# =========================
def plot_reliability_seed_panels(id_dataset, shot):
    if calibration_bins.empty:
        return None
    fig, axes = plt.subplots(1, len(SEEDS), figsize=(5.0 * len(SEEDS), 4.2), sharex=True, sharey=True)
    if len(SEEDS) == 1:
        axes = [axes]
    any_plotted = False
    for ax, seed in zip(axes, SEEDS):
        for method, linestyle in [("BayesRTMMRL", "-"), ("BayesAdapter", "--")]:
            g = calibration_bins[(calibration_bins["id_dataset"] == id_dataset) & (calibration_bins["shot"] == int(shot)) & (calibration_bins["seed"] == int(seed)) & (calibration_bins["method"] == method)]
            g = g.dropna(subset=["confidence", "accuracy"])
            if g.empty:
                continue
            ax.plot(g["confidence"], g["accuracy"], marker="o", linestyle=linestyle, label=method)
            any_plotted = True
        ax.plot([0, 1], [0, 1], linestyle=":", linewidth=1.0, label="perfect" if seed == SEEDS[0] else None)
        ax.set_title(f"seed={seed}")
        ax.set_xlabel("MSP confidence")
        ax.set_ylabel("Accuracy")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.25)
    axes[0].legend(fontsize=8)
    fig.suptitle(f"Reliability diagram: {id_dataset}, shot={shot}")
    fig.tight_layout()
    if not any_plotted:
        plt.close(fig)
        return None
    out_path = FIG_DIRS["calibration"] / f"reliability_{id_dataset}_shot{shot}_seed_panels.png"
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    return str(out_path)

rel_fig_rows = []
for id_dataset in ID_DATASETS:
    for shot in SHOTS:
        p = plot_reliability_seed_panels(id_dataset, shot)
        if p:
            rel_fig_rows.append({"id_dataset": id_dataset, "shot": shot, "figure": p})
rel_fig_df = pd.DataFrame(rel_fig_rows)
rel_fig_csv = SUMMARY_ROOT / "reliability_figures.csv"
rel_fig_df.to_csv(rel_fig_csv, index=False)
print("Saved:", rel_fig_csv)
display(rel_fig_df.head(20))

,id_dataset,shot,figure
0,cifar_10,8,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 19. Risk-Coverage Curve / AURC

基于 ID test set：按 MSP uncertainty 从低到高排序，保留低不确定样本，计算 coverage 与 risk。

In [21]:

# =========================
# 19. Risk-coverage
# =========================
def risk_coverage_curve(correct_bool, uncertainty):
    correct_bool = np.asarray(correct_bool).astype(bool)
    uncertainty = np.asarray(uncertainty, dtype=float)
    order = np.argsort(uncertainty)  # low uncertainty retained first
    corr_sorted = correct_bool[order]
    n = len(corr_sorted)
    if n == 0:
        return np.array([]), np.array([])
    coverages = np.arange(1, n + 1) / n
    errors = (~corr_sorted).astype(float)
    risks = np.cumsum(errors) / np.arange(1, n + 1)
    return coverages, risks


def aurc_eaurc(correct_bool, uncertainty):
    cov, risk = risk_coverage_curve(correct_bool, uncertainty)
    if len(cov) == 0:
        return np.nan, np.nan
    aurc = float(np.trapz(risk, cov))
    # Simplified excess AURC: subtract oracle risk-coverage integral.
    errors = (~np.asarray(correct_bool).astype(bool)).astype(float)
    oracle_order = np.argsort(errors)  # correct first, errors last
    oracle_corr = np.asarray(correct_bool).astype(bool)[oracle_order]
    oracle_cov = np.arange(1, len(oracle_corr) + 1) / len(oracle_corr)
    oracle_risk = np.cumsum((~oracle_corr).astype(float)) / np.arange(1, len(oracle_corr) + 1)
    oracle_aurc = float(np.trapz(oracle_risk, oracle_cov))
    return aurc, aurc - oracle_aurc

risk_rows = []
for id_dataset in ID_DATASETS:
    for shot in SHOTS:
        for seed in SEEDS:
            for method in ["BayesRTMMRL", "BayesAdapter"]:
                cache_dir = build_case_cache_dir(method, id_dataset, shot, seed)
                id_path = cache_dir / "id_test_outputs.pt"
                if not id_path.exists():
                    continue
                payload = load_tensor_payload(id_path)
                correct = get_correct(payload)
                unc = get_unc(payload)
                if correct is None or len(correct) != len(unc):
                    continue
                aurc, eaurc = aurc_eaurc(correct, unc)
                risk_rows.append({
                    "method": method,
                    "id_dataset": id_dataset,
                    "shot": int(shot),
                    "seed": int(seed),
                    "score_name": "msp_uncertainty",
                    "AURC": aurc,
                    "EAURC": eaurc,
                })

risk_raw = pd.DataFrame(risk_rows)
risk_raw_csv = SUMMARY_ROOT / "summary_risk_coverage_msp_raw.csv"
risk_raw.to_csv(risk_raw_csv, index=False)

risk_mean = mean_std_summary(
    risk_raw.assign(ood_dataset="", status="ok"),
    ["AURC", "EAURC"]
) if not risk_raw.empty else pd.DataFrame()
risk_mean_csv = SUMMARY_ROOT / "summary_risk_coverage_msp_mean_std.csv"
risk_mean.to_csv(risk_mean_csv, index=False)

print("Saved:", risk_raw_csv)
print("Saved:", risk_mean_csv)
display(risk_raw.head(20))

/tmp/ipykernel_12854/225006106.py:22: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  aurc = float(np.trapz(risk, cov))
/tmp/ipykernel_12854/225006106.py:29: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  oracle_aurc = float(np.trapz(oracle_risk, oracle_cov))


,method,id_dataset,shot,seed,score_name,AURC,EAURC
0,BayesRTMMRL,cifar_10,8,1,msp_uncertainty,0.008876,0.006932
1,BayesAdapter,cifar_10,8,1,msp_uncertainty,0.011279,0.008323


In [22]:

# =========================
# 19b. Risk-coverage figures
# =========================
def plot_risk_coverage_seed_panels(id_dataset, shot):
    fig, axes = plt.subplots(1, len(SEEDS), figsize=(5.0 * len(SEEDS), 4.2), sharex=True, sharey=True)
    if len(SEEDS) == 1:
        axes = [axes]
    any_plotted = False
    for ax, seed in zip(axes, SEEDS):
        for method, linestyle in [("BayesRTMMRL", "-"), ("BayesAdapter", "--")]:
            cache_dir = build_case_cache_dir(method, id_dataset, shot, seed)
            id_path = cache_dir / "id_test_outputs.pt"
            if not id_path.exists():
                continue
            payload = load_tensor_payload(id_path)
            correct = get_correct(payload)
            unc = get_unc(payload)
            if correct is None or len(correct) != len(unc):
                continue
            cov, risk = risk_coverage_curve(correct, unc)
            ax.plot(cov, risk, linestyle=linestyle, label=method)
            any_plotted = True
        ax.set_title(f"seed={seed}")
        ax.set_xlabel("Coverage")
        ax.set_ylabel("Risk")
        ax.set_xlim(0, 1)
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.25)
    axes[0].legend(fontsize=8)
    fig.suptitle(f"Risk-Coverage Curve: {id_dataset}, shot={shot}, score=MSP uncertainty")
    fig.tight_layout()
    if not any_plotted:
        plt.close(fig)
        return None
    out_path = FIG_DIRS["risk_coverage"] / f"risk_coverage_{id_dataset}_shot{shot}_seed_panels.png"
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    return str(out_path)

risk_fig_rows = []
for id_dataset in ID_DATASETS:
    for shot in SHOTS:
        p = plot_risk_coverage_seed_panels(id_dataset, shot)
        if p:
            risk_fig_rows.append({"id_dataset": id_dataset, "shot": shot, "figure": p})
risk_fig_df = pd.DataFrame(risk_fig_rows)
risk_fig_csv = SUMMARY_ROOT / "risk_coverage_figures.csv"
risk_fig_df.to_csv(risk_fig_csv, index=False)
print("Saved:", risk_fig_csv)
display(risk_fig_df.head(20))

,id_dataset,shot,figure
0,cifar_10,8,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 20. MSP OOD metric delta 图

这里画基于 MSP uncertainty 的 OOD 指标 delta。

`delta = BayesRTMMRL - BayesAdapter`，但不同指标方向不同，因此图表索引里保留 `metric_direction`。

In [23]:

# =========================
# 20. Delta metric figures
# =========================
def plot_delta_metric(delta_df: pd.DataFrame, metric: str):
    if delta_df.empty:
        return None
    ok = delta_df[(delta_df["status"] == "ok") & (delta_df["metric"] == metric)].copy()
    if ok.empty:
        return None
    ok["case"] = ok["id_dataset"].astype(str) + "→" + ok["ood_dataset"].astype(str) + " s" + ok["shot"].astype(str)
    ok = ok.sort_values(["id_dataset", "ood_dataset", "shot"])
    col = "delta_BayesRTMMRL_minus_BayesAdapter"
    fig_path = FIG_DIRS["metric_delta"] / f"delta_msp_{metric}.png"
    plt.figure(figsize=(max(8, len(ok) * 0.35), 4.6))
    plt.bar(np.arange(len(ok)), ok[col].astype(float).to_numpy())
    plt.axhline(0.0, linewidth=1)
    plt.xticks(np.arange(len(ok)), ok["case"].tolist(), rotation=70, ha="right")
    plt.ylabel(f"BayesRTMMRL - BayesAdapter ({metric})")
    plt.title(f"OOD delta based on MSP uncertainty: {metric} ({metric_direction(metric)})")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=220)
    plt.close()
    return str(fig_path)

delta_fig_rows = []
for metric in ["AUROC", "AUPR_OUT", "AUPR_IN", "FPR95", "DetectionError"]:
    p = plot_delta_metric(ood_delta, metric)
    if p:
        delta_fig_rows.append({"metric": metric, "metric_direction": metric_direction(metric), "figure": p})

delta_fig_df = pd.DataFrame(delta_fig_rows)
delta_fig_csv = SUMMARY_ROOT / "delta_msp_figures.csv"
delta_fig_df.to_csv(delta_fig_csv, index=False)
print("Saved:", delta_fig_csv)
display(delta_fig_df)

,metric,metric_direction,figure
0,AUROC,higher_better,/root/autodl-tmp/MMRL/output_refactor/analysis...
1,AUPR_OUT,higher_better,/root/autodl-tmp/MMRL/output_refactor/analysis...
2,AUPR_IN,higher_better,/root/autodl-tmp/MMRL/output_refactor/analysis...
3,FPR95,lower_better,/root/autodl-tmp/MMRL/output_refactor/analysis...
4,DetectionError,lower_better,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 21. 输出索引

最终索引会记录所有 summary、figure index 和 cache root。

In [24]:

# =========================
# 21. 输出索引
# =========================
outputs = {
    "notebook_version": NOTEBOOK_VERSION,
    "analysis_root": str(ANALYSIS_ROOT),
    "cache_root": str(CACHE_ROOT),
    "prediction_cache_root": str(PRED_CACHE_ROOT),
    "feature_cache_root": str(FEATURE_CACHE_ROOT),
    "summary_root": str(SUMMARY_ROOT),
    "figure_root": str(FIGURE_ROOT),
    "paper_ready_root": str(PAPER_READY_ROOT),
    "main_uncertainty_score": MAIN_UNCERTAINTY_SCORE,
    "summary_ood_msp_raw": str(SUMMARY_ROOT / "summary_ood_msp_raw.csv"),
    "summary_ood_msp_mean_std": str(SUMMARY_ROOT / "summary_ood_msp_mean_std.csv"),
    "summary_ood_msp_delta": str(SUMMARY_ROOT / "summary_ood_msp_delta.csv"),
    "summary_similarity_raw": str(SUMMARY_ROOT / "summary_similarity_raw.csv"),
    "summary_similarity_mean_std": str(SUMMARY_ROOT / "summary_similarity_mean_std.csv"),
    "summary_similarity_delta_vs_adapter": str(SUMMARY_ROOT / "summary_similarity_delta_vs_adapter.csv"),
    "summary_calibration_raw": str(SUMMARY_ROOT / "summary_calibration_raw.csv"),
    "summary_calibration_mean_std": str(SUMMARY_ROOT / "summary_calibration_mean_std.csv"),
    "summary_risk_coverage_msp_raw": str(SUMMARY_ROOT / "summary_risk_coverage_msp_raw.csv"),
    "summary_risk_coverage_msp_mean_std": str(SUMMARY_ROOT / "summary_risk_coverage_msp_mean_std.csv"),
    "figure_indices": {
        "msp_uncertainty_distribution": str(SUMMARY_ROOT / "msp_uncertainty_distribution_figures.csv"),
        "msp_correct_wrong_ood": str(SUMMARY_ROOT / "msp_correct_wrong_ood_figures.csv"),
        "feature_embedding_R_C_adapter": str(SUMMARY_ROOT / "feature_embedding_R_C_adapter_figures.csv"),
        "similarity_matrix_R_C_adapter": str(SUMMARY_ROOT / "similarity_matrix_R_C_adapter_figures.csv"),
        "similarity_distribution_R_C_adapter": str(SUMMARY_ROOT / "similarity_distribution_R_C_adapter_figures.csv"),
        "reliability": str(SUMMARY_ROOT / "reliability_figures.csv"),
        "risk_coverage": str(SUMMARY_ROOT / "risk_coverage_figures.csv"),
        "delta_msp": str(SUMMARY_ROOT / "delta_msp_figures.csv"),
    },
    "figure_dirs": {k: str(v) for k, v in FIG_DIRS.items()},
}

index_path = SUMMARY_ROOT / "output_index.json"
with index_path.open("w", encoding="utf-8") as f:
    json.dump(outputs, f, indent=2, ensure_ascii=False)

print(json.dumps(outputs, indent=2, ensure_ascii=False))